# FP Analysis

In [ ]:
# ============================================================
# FULLY UNSUPERVISED PIPELINE - NO LABELS USED
# 4 LLMs VERSION: Llama, Qwen, Mistral, Qwen32B
# ============================================================
"""
Fully unsupervised pipeline for legal ranking.

TIE-BREAK HANDLING:
====================================
The binary methods (LLM_Union, LLM_Inter*, etc.) produce
binary scores (0 or 1), creating many ties. To break them:

  Score final = Score_LLM + ε × Score_CrossEncoder

where ε = 1e-6, small enough for the LLM score to dominate.

COMBINATIONS TESTED (4 LLMs):
==============================
- Union: at least 1 LLM says yes
- Inter2: at least 2 LLMs say yes
- Inter3_all: the 4 combinations of 3 LLMs:
    * Inter3_LQM: Llama + Qwen + Mistral
    * Inter3_LQQ32: Llama + Qwen + Qwen32B
    * Inter3_LMQ32: Llama + Mistral + Qwen32B
    * Inter3_QMQ32: Qwen + Mistral + Qwen32B
- Inter3: at least 3 of the 4 LLMs say yes
- Inter4: all 4 LLMs say yes

Reference: Voorhees (2000), TREC evaluation methodology
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, field
import json
from itertools import combinations

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# DEPENDENCY INSTALLATION
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess
    packages = [
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]
    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    qwen32b_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    # Targets split by agreement/disagreement
    targets_by_agreement: List[str] = field(default_factory=lambda: [
        "Gold_Agree", "Gold_Disagree",
        "A1_Agree", "A1_Disagree",
        "A2_Agree", "A2_Disagree"
    ])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    random_seed: int = 42

    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.qwen32b_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-32b.xlsx"
        self.output_dir = f"artifacts/outputs_unsupervised_4llm"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        dcg = float(np.sum(y_sorted / np.log2(np.arange(2, k + 2))))

        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:k]
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k + 2))))

        return dcg / idcg if idcg > 0 else 0.0

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0


# ============================================================
# DATA LOADING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 4 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA (4 LLMs)")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)
    df_qwen32b = pd.read_excel(config.qwen32b_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)} | Qwen32B: {len(df_qwen32b)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]
    for df in [df_main, df_llama, df_qwen, df_mistral, df_qwen32b]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    # Llama
    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    # Qwen (7B)
    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    # Mistral
    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    # Qwen 32B
    pred_col_q32 = None
    for col in df_qwen32b.columns:
        if "pred" in col.lower() and col not in key_cols:
            pred_col_q32 = col
            break

    if pred_col_q32 is None:
        for col in df_qwen32b.columns:
            if col not in key_cols and col != "_key":
                vals = df_qwen32b[col].dropna().unique()
                if set(vals).issubset({0, 1, "0", "1", "oui", "non", "Oui", "Non"}):
                    pred_col_q32 = col
                    break

    if pred_col_q32:
        df_qwen32b = df_qwen32b.rename(columns={pred_col_q32: "Qwen32B_Pred"})
    else:
        df_qwen32b["Qwen32B_Pred"] = 0
        print("  WARNING: Could not find prediction column for Qwen32B, defaulting to 0")

    qwen32b_cols = df_qwen32b[["_key", "Qwen32B_Pred"]].copy()

    # Merge all
    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")
    df = df.merge(qwen32b_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)

    # Build labels for agree/disagree cases
    # Agree = A2 and A1 concur (same value, both non-NaN)
    # Disagree = A2 and A1 differ (different values, both non-NaN)
    df["is_agree"] = (df["lbl_A2"] == df["lbl_A1"]) & df["lbl_A2"].notna() & df["lbl_A1"].notna()
    df["is_disagree"] = (df["lbl_A2"] != df["lbl_A1"]) & df["lbl_A2"].notna() & df["lbl_A1"].notna()

    # Labels filtered by agreement/disagreement
    df["lbl_Gold_Agree"] = df.apply(lambda r: r["lbl_Gold"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_Gold_Disagree"] = df.apply(lambda r: r["lbl_Gold"] if r["is_disagree"] else np.nan, axis=1)
    df["lbl_A1_Agree"] = df.apply(lambda r: r["lbl_A1"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_A1_Disagree"] = df.apply(lambda r: r["lbl_A1"] if r["is_disagree"] else np.nan, axis=1)
    df["lbl_A2_Agree"] = df.apply(lambda r: r["lbl_A2"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_A2_Disagree"] = df.apply(lambda r: r["lbl_A2"] if r["is_disagree"] else np.nan, axis=1)

    # Agree/disagree stats
    n_agree = df["is_agree"].sum()
    n_disagree = df["is_disagree"].sum()
    print(f"\n  Agreement/Disagreement between annotators:")
    print(f"    - Agree (A2 == A1):   {n_agree} ({n_agree/len(df)*100:.1f}%)")
    print(f"    - Disagree (A2 != A1): {n_disagree} ({n_disagree/len(df)*100:.1f}%)")

    df = df.drop(columns=["_key"])

    return df


# ============================================================
# FEATURES
# ============================================================
def compute_tfidf_similarity(df: pd.DataFrame) -> np.ndarray:
    """TF-IDF similarity."""
    stopwords_fr = [
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ]

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2,
        max_df=0.95, stop_words=stopwords_fr, lowercase=True
    )
    vectorizer.fit(texts + articles)

    tfidf_texts = vectorizer.transform(texts)
    tfidf_articles = vectorizer.transform(articles)

    return np.array([
        cosine_similarity(tfidf_texts[i], tfidf_articles[i])[0, 0]
        for i in range(len(texts))
    ])


def compute_bm25_scores(df: pd.DataFrame) -> np.ndarray:
    """BM25 scores."""
    from rank_bm25 import BM25Okapi

    stopwords = set(["le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "à"])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords and len(t) > 2]

    articles = df["article_text"].fillna("").tolist()
    texts = df["text"].fillna("").tolist()

    tokenized_articles = [tokenize(a) for a in articles]
    bm25 = BM25Okapi(tokenized_articles)

    scores = []
    for i, text in enumerate(texts):
        query = tokenize(text)
        if query:
            all_scores = bm25.get_scores(query)
            scores.append(all_scores[i])
        else:
            scores.append(0.0)

    scores = np.array(scores)
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores


def compute_cross_encoder_scores(df: pd.DataFrame, config: Config) -> np.ndarray:
    """Cross-encoder 0-shot."""
    print("\n  Computing Cross-Encoder scores...")

    from sentence_transformers import CrossEncoder
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"    Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()
    pairs = [[t, a] for t, a in zip(texts, articles)]

    scores = []
    batch_size = config.cross_encoder_batch_size

    for i in tqdm(range(0, len(pairs), batch_size), desc="    Cross-encoder"):
        batch = pairs[i:i + batch_size]
        batch_scores = model.predict(batch, show_progress_bar=False)
        scores.extend(batch_scores)

    scores = np.array(scores)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    print(f"    Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    return scores_normalized


def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all UNSUPERVISED features with 4 LLMs."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION (UNSUPERVISED - 4 LLMs)")
    print("="*70)

    df = df.copy()

    # Individual votes of the 4 LLMs
    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)
    df["vote_qwen32b"] = (df["Qwen32B_Pred"] == 1).astype(int)

    # Sum of votes (0-4)
    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"] + df["vote_qwen32b"]

    # Union: at least 1 LLM says yes
    df["vote_union"] = (df["vote_sum"] >= 1).astype(int)

    # Inter2: at least 2 LLMs say yes
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)

    # Inter3: at least 3 of the 4 LLMs say yes
    df["vote_inter3"] = (df["vote_sum"] >= 3).astype(int)

    # Inter4: all 4 LLMs say yes
    df["vote_inter4"] = (df["vote_sum"] == 4).astype(int)

    # The 4 specific combinations of 3 LLMs (exact intersection of 3 specific LLMs)
    # L = Llama, Q = Qwen, M = Mistral, Q32 = Qwen32B
    df["vote_inter3_LQM"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) & (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter3_LQQ32"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_LMQ32"] = ((df["vote_llama"] == 1) & (df["vote_mistral"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_QMQ32"] = ((df["vote_qwen"] == 1) & (df["vote_mistral"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)

    print("  [1/3] TF-IDF...")
    df["sim_tfidf"] = compute_tfidf_similarity(df)

    print("  [2/3] BM25...")
    df["sim_bm25"] = compute_bm25_scores(df)

    if compute_crossencoder:
        print("  [3/3] Cross-Encoder...")
        df["sim_crossencoder"] = compute_cross_encoder_scores(df, config)
    else:
        df["sim_crossencoder"] = 0.0

    return df


# ============================================================
# RANKING SCORES (with integrated tie-break)
# ============================================================
def compute_all_ranking_scores(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    """
    Compute ranking scores for all methods.

    TIE-BREAK HANDLING:
    - Continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic): direct scores
    - Binary methods (LLM_*): score = vote + ε × CrossEncoder to break ties
    """
    scores = {}

    # Continuous methods - no tie-break needed
    scores["TF-IDF"] = df["sim_tfidf"].values
    scores["BM25"] = df["sim_bm25"].values
    scores["CrossEncoder"] = df["sim_crossencoder"].values

    # Heuristic - weighted combination (near-unique scores) - updated for 4 LLMs
    weights = {
        "vote_llama": 0.25, "vote_qwen": 0.25, "vote_mistral": 0.25, "vote_qwen32b": 0.25,
        "vote_inter2": 0.3, "vote_inter3": 0.5, "vote_inter4": 1.0,
        "sim_tfidf": 0.2, "sim_bm25": 0.2, "sim_crossencoder": 0.4,
    }
    heuristic = np.zeros(len(df))
    for feature, weight in weights.items():
        if feature in df.columns:
            heuristic += weight * df[feature].values
    scores["Heuristic"] = heuristic

    # Binary methods - tie-break via CrossEncoder
    # Score = vote + ε × CrossEncoder_normalized
    eps = 1e-6
    ce_norm = df["sim_crossencoder"].values
    if ce_norm.max() > ce_norm.min():
        ce_norm = (ce_norm - ce_norm.min()) / (ce_norm.max() - ce_norm.min())

    # Main methods
    scores["LLM_Union"] = df["vote_union"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter2"] = df["vote_inter2"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter3"] = df["vote_inter3"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter4"] = df["vote_inter4"].values.astype(float) + eps * ce_norm

    # The 4 specific combinations of 3 LLMs
    scores["Inter3_LQM"] = df["vote_inter3_LQM"].values.astype(float) + eps * ce_norm
    scores["Inter3_LQQ32"] = df["vote_inter3_LQQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_LMQ32"] = df["vote_inter3_LMQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_QMQ32"] = df["vote_inter3_QMQ32"].values.astype(float) + eps * ce_norm

    return scores


# ============================================================
# EVALUATION (with count and ratio)
# ============================================================
def evaluate_method(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate a ranking with the number of positives retrieved."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            count_k = Metrics.count_at_k(y, s, k)
            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count": count_k,
                "total_positives": n_pos,
                "count_ratio": f"{count_k}/{n_pos}",
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """Random baseline = prevalence."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        for k in config.k_values:
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,
                "count": int(expected_pos_at_k),
                "total_positives": n_pos,
                "count_ratio": f"{int(expected_pos_at_k)}/{n_pos}",
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# COVERAGE ANALYSIS
# ============================================================
def analyze_positive_coverage(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Analysis: how many positives retrieved at each k."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))

        if n_pos == 0:
            continue

        prevalence = n_pos / n_samples
        order = np.argsort(s)[::-1]
        y_sorted = y[order]

        for k in config.k_values:
            k_actual = min(k, n_samples)
            positives_at_k = int(np.sum(y_sorted[:k_actual]))
            random_expected = k_actual * prevalence

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "positives_retrieved": positives_at_k,
                "total_positives": n_pos,
                "coverage": positives_at_k / n_pos,
                "count_ratio": f"{positives_at_k}/{n_pos}",
                "random_expected": random_expected,
                "delta_vs_random": positives_at_k - random_expected,
                "n_samples": n_samples,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def analyze_llm_intersection_coverage(df: pd.DataFrame) -> Dict:
    """Coverage analysis by LLM intersections (4 LLMs)."""
    stats = {
        "inter4": int((df["vote_inter4"] == 1).sum()),
        "inter3": int((df["vote_inter3"] == 1).sum()),
        "inter3_LQM": int((df["vote_inter3_LQM"] == 1).sum()),
        "inter3_LQQ32": int((df["vote_inter3_LQQ32"] == 1).sum()),
        "inter3_LMQ32": int((df["vote_inter3_LMQ32"] == 1).sum()),
        "inter3_QMQ32": int((df["vote_inter3_QMQ32"] == 1).sum()),
        "inter2": int((df["vote_inter2"] == 1).sum()),
        "union": int((df["vote_union"] == 1).sum()),
        "no_vote": int((df["vote_union"] == 0).sum()),
        "total": int(len(df))
    }

    # Per-LLM stats
    stats["llama_yes"] = int((df["vote_llama"] == 1).sum())
    stats["qwen_yes"] = int((df["vote_qwen"] == 1).sum())
    stats["mistral_yes"] = int((df["vote_mistral"] == 1).sum())
    stats["qwen32b_yes"] = int((df["vote_qwen32b"] == 1).sum())

    for target in ["A1", "A2", "Gold"]:
        label_col = f"lbl_{target}"
        if label_col not in df.columns:
            continue

        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        df_target = df[mask].copy()
        y = df_target[label_col].astype(int)

        stats[f"{target}_total"] = int(len(y))
        stats[f"{target}_positives"] = int(y.sum())
        stats[f"{target}_pos_inter4"] = int(y[df_target["vote_inter4"] == 1].sum())
        stats[f"{target}_pos_inter3"] = int(y[df_target["vote_inter3"] == 1].sum())
        stats[f"{target}_pos_inter3_LQM"] = int(y[df_target["vote_inter3_LQM"] == 1].sum())
        stats[f"{target}_pos_inter3_LQQ32"] = int(y[df_target["vote_inter3_LQQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_LMQ32"] = int(y[df_target["vote_inter3_LMQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_QMQ32"] = int(y[df_target["vote_inter3_QMQ32"] == 1].sum())
        stats[f"{target}_pos_inter2"] = int(y[df_target["vote_inter2"] == 1].sum())
        stats[f"{target}_pos_union"] = int(y[df_target["vote_union"] == 1].sum())
        stats[f"{target}_pos_no_vote"] = int(y[df_target["vote_union"] == 0].sum())

    return stats


def analyze_agree_disagree_performance(
    df: pd.DataFrame,
    all_scores: Dict[str, np.ndarray],
    config: Config
) -> pd.DataFrame:
    """Analyze performance on agree vs disagree cases."""
    results = []

    # Targets to analyze
    targets_agreement = ["Gold_Agree", "Gold_Disagree", "A1_Agree", "A1_Disagree",
                         "A2_Agree", "A2_Disagree"]

    for method_name, scores in all_scores.items():
        for target in targets_agreement:
            label_col = f"lbl_{target}"
            if label_col not in df.columns:
                continue

            mask = df[label_col].notna()
            if mask.sum() == 0:
                continue

            y = df.loc[mask, label_col].values.astype(int)
            s = scores[mask.values]

            n_samples = len(y)
            n_pos = int(np.sum(y))
            if n_samples == 0:
                continue

            prevalence = n_pos / n_samples
            ap = Metrics.average_precision(y, s)

            # Extraire base_target et agreement_type
            parts = target.rsplit('_', 1)
            base_target = parts[0]
            agreement_type = parts[1] if len(parts) > 1 else "All"

            for k in config.k_values:
                count_k = Metrics.count_at_k(y, s, k)
                results.append({
                    "method": method_name,
                    "target": target,
                    "base_target": base_target,
                    "agreement": agreement_type,
                    "k": k,
                    "P@k": Metrics.precision_at_k(y, s, k),
                    "R@k": Metrics.recall_at_k(y, s, k),
                    "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                    "count": count_k,
                    "total_positives": n_pos,
                    "count_ratio": f"{count_k}/{n_pos}",
                    "AP": ap,
                    "n_samples": n_samples,
                    "prevalence": prevalence
                })

    return pd.DataFrame(results)


def analyze_false_positives(
    df: pd.DataFrame,
    all_scores: Dict[str, np.ndarray],
    config: Config
) -> pd.DataFrame:
    """
    Detailed false-positive analysis for each method.

    For each k, compute:
    - TP (True Positives): documents correctly identified as positive
    - FP (False Positives): documents incorrectly identified as positive
    - FP_Agree: FP on cases where annotators agreed (clear error)
    - FP_Disagree: FP on cases where annotators disagreed (gray zone)
    """
    results = []

    for method_name, scores in all_scores.items():
        for target in config.targets:
            label_col = f"lbl_{target}"
            mask = df[label_col].notna()

            if mask.sum() == 0:
                continue

            df_subset = df[mask].copy()
            y = df_subset[label_col].values.astype(int)
            s = scores[mask.values]

            n_samples = len(y)
            n_pos = int(np.sum(y))
            n_neg = n_samples - n_pos

            # Sort by descending score
            order = np.argsort(s)[::-1]
            y_sorted = y[order]
            df_sorted = df_subset.iloc[order].copy()

            for k in config.k_values:
                k_actual = min(k, n_samples)

                # Top-k predictions
                y_topk = y_sorted[:k_actual]
                df_topk = df_sorted.iloc[:k_actual]

                # Basic computations
                tp = int(np.sum(y_topk == 1))
                fp = int(np.sum(y_topk == 0))

                # Precision and recall
                precision = tp / k_actual if k_actual > 0 else 0
                recall = tp / n_pos if n_pos > 0 else 0

                # Analyze FP by type (Agree vs Disagree)
                fp_agree = 0
                fp_disagree = 0

                if "is_agree" in df_topk.columns and "is_disagree" in df_topk.columns:
                    # FP = documents predicted positive (in top-k) but label = 0
                    fp_mask = (y_topk == 0)
                    fp_indices = df_topk.index[fp_mask]

                    for idx in fp_indices:
                        if df.loc[idx, "is_agree"]:
                            fp_agree += 1
                        elif df.loc[idx, "is_disagree"]:
                            fp_disagree += 1

                # FP rate
                fp_rate = fp / k_actual if k_actual > 0 else 0
                fp_agree_rate = fp_agree / fp if fp > 0 else 0
                fp_disagree_rate = fp_disagree / fp if fp > 0 else 0

                # Comparison with random baseline
                random_fp = k_actual * (n_neg / n_samples)
                fp_reduction = (random_fp - fp) / random_fp if random_fp > 0 else 0

                results.append({
                    "method": method_name,
                    "target": target,
                    "k": k,
                    "TP": tp,
                    "FP": fp,
                    "FP_Agree": fp_agree,
                    "FP_Disagree": fp_disagree,
                    "Precision": precision,
                    "Recall": recall,
                    "FP_Rate": fp_rate,
                    "FP_Agree_Rate": fp_agree_rate,
                    "FP_Disagree_Rate": fp_disagree_rate,
                    "Random_FP_Expected": random_fp,
                    "FP_Reduction_vs_Random": fp_reduction,
                    "n_positives": n_pos,
                    "n_negatives": n_neg,
                    "n_samples": n_samples
                })

    return pd.DataFrame(results)


def plot_false_positive_analysis(df_fp: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot of the false-positive analysis."""
    if df_fp.empty:
        return

    # Focus on Gold
    df_gold = df_fp[df_fp["target"] == "Gold"]
    if df_gold.empty:
        return

    methods_to_plot = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2", "LLM_Union", "CrossEncoder"]
    methods_to_plot = [m for m in methods_to_plot if m in df_gold["method"].unique()]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter4': 'brown', 'LLM_Inter3': 'purple',
        'LLM_Inter2': 'red', 'LLM_Union': 'orange', 'CrossEncoder': 'green'
    }

    # Plot 1: Number of FP vs k
    ax = axes[0]
    for method in methods_to_plot:
        df_m = df_gold[df_gold["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    # Random baseline
    df_first = df_gold[df_gold["method"] == methods_to_plot[0]].sort_values("k")
    if not df_first.empty:
        ax.plot(df_first["k"], df_first["Random_FP_Expected"],
               color='black', linestyle='--', linewidth=1.5, label='Random (expected)', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("Number of False Positives")
    ax.set_title("Gold - False Positives vs k")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

    # Plot 2: FP reduction vs Random
    ax = axes[1]
    for method in methods_to_plot:
        df_m = df_gold[df_gold["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP_Reduction_vs_Random"] * 100,
                   color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel("k")
    ax.set_ylabel("FP Reduction vs Random (%)")
    ax.set_title("Gold - False-Positive Reduction")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

    # Plot 3: FP Disagree proportion
    ax = axes[2]
    for method in methods_to_plot:
        df_m = df_gold[df_gold["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP_Disagree_Rate"] * 100,
                   color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    ax.set_xlabel("k")
    ax.set_ylabel("% of FP on Disagree cases")
    ax.set_title("Gold - Proportion of FP in the gray zone")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# REPORTING (with count/total)
# ============================================================
def print_llm_intersection_analysis(stats: Dict):
    """Print the LLM intersection analysis (4 LLMs)."""
    print("\n" + "="*70)
    print("LLM INTERSECTION ANALYSIS (4 LLMs)")
    print("="*70)

    print(f"\n  Full dataset: {stats['total']} entries")
    print(f"\n  Individual votes:")
    print(f"    - Llama:   {stats['llama_yes']:>6} ({stats['llama_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen:    {stats['qwen_yes']:>6} ({stats['qwen_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Mistral: {stats['mistral_yes']:>6} ({stats['mistral_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen32B: {stats['qwen32b_yes']:>6} ({stats['qwen32b_yes']/stats['total']*100:>5.1f}%)")

    print(f"\n  Intersections:")
    print(f"    - 4 LLMs agree (inter4):  {stats['inter4']:>6} ({stats['inter4']/stats['total']*100:>5.1f}%)")
    print(f"    - 3+ LLMs agree (inter3): {stats['inter3']:>6} ({stats['inter3']/stats['total']*100:>5.1f}%)")
    print(f"    - 2+ LLMs agree (inter2): {stats['inter2']:>6} ({stats['inter2']/stats['total']*100:>5.1f}%)")
    print(f"    - 1+ LLM agrees (union):   {stats['union']:>6} ({stats['union']/stats['total']*100:>5.1f}%)")
    print(f"    - No LLM agrees:        {stats['no_vote']:>6} ({stats['no_vote']/stats['total']*100:>5.1f}%)")

    print(f"\n  Specific combinations of 3 LLMs:")
    print(f"    - Llama+Qwen+Mistral (LQM):     {stats['inter3_LQM']:>6} ({stats['inter3_LQM']/stats['total']*100:>5.1f}%)")
    print(f"    - Llama+Qwen+Qwen32B (LQQ32):   {stats['inter3_LQQ32']:>6} ({stats['inter3_LQQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - Llama+Mistral+Qwen32B (LMQ32):{stats['inter3_LMQ32']:>6} ({stats['inter3_LMQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen+Mistral+Qwen32B (QMQ32): {stats['inter3_QMQ32']:>6} ({stats['inter3_QMQ32']/stats['total']*100:>5.1f}%)")

    for target in ["A1", "A2", "Gold"]:
        if f"{target}_total" not in stats:
            continue

        n_total = stats[f"{target}_total"]
        n_pos = stats[f"{target}_positives"]

        if n_pos == 0:
            continue

        print(f"\n  {target}: {n_pos} positives out of {n_total} ({n_pos/n_total*100:.1f}%)")
        print(f"    - In inter4:     {stats[f'{target}_pos_inter4']:>4}/{n_pos} ({stats[f'{target}_pos_inter4']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter3:     {stats[f'{target}_pos_inter3']:>4}/{n_pos} ({stats[f'{target}_pos_inter3']/n_pos*100:>5.1f}% of positives)")
        print(f"      * LQM:           {stats[f'{target}_pos_inter3_LQM']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LQM']/n_pos*100:>5.1f}%)")
        print(f"      * LQQ32:         {stats[f'{target}_pos_inter3_LQQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LQQ32']/n_pos*100:>5.1f}%)")
        print(f"      * LMQ32:         {stats[f'{target}_pos_inter3_LMQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LMQ32']/n_pos*100:>5.1f}%)")
        print(f"      * QMQ32:         {stats[f'{target}_pos_inter3_QMQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_QMQ32']/n_pos*100:>5.1f}%)")
        print(f"    - In inter2:     {stats[f'{target}_pos_inter2']:>4}/{n_pos} ({stats[f'{target}_pos_inter2']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In union:      {stats[f'{target}_pos_union']:>4}/{n_pos} ({stats[f'{target}_pos_union']/n_pos*100:>5.1f}% of positives)")
        print(f"    - Outside union:      {stats[f'{target}_pos_no_vote']:>4}/{n_pos} ({stats[f'{target}_pos_no_vote']/n_pos*100:>5.1f}% of positives)")


def print_comparison_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Comparison table with count/total."""
    print("\n" + "="*160)
    print("COMPARISON OF UNSUPERVISED METHODS (4 LLMs) - with num yes retrieved / num yes total")
    print("="*160)

    # Main methods only for the first table
    main_methods = ["Random", "TF-IDF", "BM25", "CrossEncoder", "Heuristic",
                   "LLM_Union", "LLM_Inter2", "LLM_Inter3", "LLM_Inter4"]
    main_methods = [m for m in main_methods if m in results_dict]

    for target in config.targets:
        print(f"\n{'='*160}")
        print(f"TARGET: {target}")
        print(f"{'='*160}")

        df_random = results_dict.get("Random")
        prevalence = None
        n_pos_total = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                n_pos_total = df_t_rand["total_positives"].iloc[0]
                print(f"  Prevalence (yes rate): {prevalence:.3f} ({prevalence*100:.1f}%)")
                print(f"  Total number of YES: {n_pos_total}")

        # Table AP
        print("\n  METHOD                | AP     | Δ vs Random")
        print("  " + "-"*50)
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = ap - prevalence
                    delta_str = f"+{delta:.3f}" if delta >= 0 else f"{delta:.3f}"
                else:
                    delta_str = "-"
                print(f"  {method:<22} | {ap:.4f} | {delta_str}")

        # P@k table for main methods
        print(f"\n  --- P@k for main methods (yes/tot) ---")
        print(f"  {'k':<6}", end="")
        for method in main_methods:
            print(f" | {method[:14]:<14}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in main_methods:
            print(f" | P@k   (yes/tot)", end="")
        print()
        print("  " + "-"*(7 + 18 * len(main_methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in main_methods:
                df_res = results_dict[method]
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    count = df_t["count"].values[0]
                    total = df_t["total_positives"].values[0]
                    row += f" | {p:.3f} ({count:>3}/{total:<3})"
                else:
                    row += " | -               "
            print(row)

        # P@k table for the 3-LLM combinations
        inter3_methods = ["Inter3_LQM", "Inter3_LQQ32", "Inter3_LMQ32", "Inter3_QMQ32"]
        inter3_methods = [m for m in inter3_methods if m in results_dict]

        if inter3_methods:
            print(f"\n  --- P@k for combinations of 3 LLMs (yes/tot) ---")
            print(f"  {'k':<6}", end="")
            for method in inter3_methods:
                print(f" | {method:<16}", end="")
            print()
            print(f"  {'':6}", end="")
            for _ in inter3_methods:
                print(f" | P@k   (yes/tot) ", end="")
            print()
            print("  " + "-"*(7 + 19 * len(inter3_methods)))

            for k in [10, 50, 100, 200, 300, 500, 1000]:
                if k not in config.k_values:
                    continue
                row = f"  {k:<6}"
                for method in inter3_methods:
                    df_res = results_dict[method]
                    df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                    if len(df_t) > 0:
                        p = df_t["P@k"].values[0]
                        count = df_t["count"].values[0]
                        total = df_t["total_positives"].values[0]
                        row += f" | {p:.3f} ({count:>3}/{total:<3}) "
                    else:
                        row += " | -                "
                print(row)


def print_coverage_table(coverage_dict: Dict[str, pd.DataFrame], config: Config):
    """Coverage table with count/total."""
    print("\n" + "="*180)
    print("POSITIVE COVERAGE PER METHOD (4 LLMs) - yes retrieved / yes total")
    print("="*180)

    # Main methods
    main_methods = ["TF-IDF", "BM25", "CrossEncoder", "Heuristic",
                   "LLM_Union", "LLM_Inter2", "LLM_Inter3", "LLM_Inter4"]
    main_methods = [m for m in main_methods if m in coverage_dict]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")

        df_first = next(iter(coverage_dict.values()))
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue

        total_pos = df_t["total_positives"].iloc[0]
        n_samples = df_t["n_samples"].iloc[0]
        prevalence = df_t["prevalence"].iloc[0]

        print(f"  Total positives (YES): {total_pos} / {n_samples} (prevalence: {prevalence:.3f})")
        print(f"{'='*180}")

        print(f"\n  {'k':<6}", end="")
        for method in main_methods:
            print(f" | {method[:16]:<16}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in main_methods:
            print(f" | yes/tot (R@k)   ", end="")
        print()
        print("  " + "-"*(7 + 19 * len(main_methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in main_methods:
                df_cov = coverage_dict[method]
                df_tk = df_cov[(df_cov["target"] == target) & (df_cov["k"] == k)]
                if len(df_tk) > 0:
                    retrieved = int(df_tk["positives_retrieved"].values[0])
                    total = int(df_tk["total_positives"].values[0])
                    recall = df_tk["coverage"].values[0]
                    row += f" | {retrieved:>3}/{total:<3} ({recall:.2f}) "
                else:
                    row += " | -                "
            print(row)


def print_detailed_count_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Detailed table with counts only (yes retrieved / yes total)."""
    print("\n" + "="*180)
    print("DETAIL: NUMBER OF YES RETRIEVED / NUMBER OF YES TOTAL (4 LLMs)")
    print("="*180)

    methods = [m for m in results_dict.keys() if m != "Random"]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")
        print(f"{'='*180}")

        # Get the total number of positives
        df_first = results_dict[methods[0]]
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue
        total_pos = df_t["total_positives"].iloc[0]
        print(f"  Total number of YES: {total_pos}")

        # Header
        print(f"\n  {'k':<6} | {'Random':<12}", end="")
        for method in methods:
            print(f" | {method[:12]:<12}", end="")
        print()
        print("  " + "-"*(7 + 15 + 15 * len(methods)))

        for k in config.k_values:
            row = f"  {k:<6}"

            # Random
            df_rand = results_dict["Random"]
            df_tk = df_rand[(df_rand["target"] == target) & (df_rand["k"] == k)]
            if len(df_tk) > 0:
                count = int(df_tk["count"].values[0])
                row += f" | {count:>4}/{total_pos:<4}   "
            else:
                row += " | -            "

            # Other methods
            for method in methods:
                df_res = results_dict[method]
                df_tk = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_tk) > 0:
                    count = int(df_tk["count"].values[0])
                    row += f" | {count:>4}/{total_pos:<4}  "
                else:
                    row += " | -            "
            print(row)


def print_agree_disagree_comparison(df_agree_disagree: pd.DataFrame, config: Config):
    """Print the performance comparison on agree vs disagree cases."""
    print("\n" + "="*180)
    print("AGREE vs DISAGREE COMPARISON - Do the methods perform differently on the hard cases?")
    print("="*180)

    if df_agree_disagree.empty:
        print("  No data available.")
        return

    # Focus on Gold (le plus pertinent)
    for base_target in ["Gold", "A1", "A2"]:
        df_base = df_agree_disagree[df_agree_disagree["base_target"] == base_target]
        if df_base.empty:
            continue

        print(f"\n{'='*180}")
        print(f"TARGET: {base_target}")
        print(f"{'='*180}")

        # Basic stats
        df_agree = df_base[df_base["agreement"] == "Agree"]
        df_disagree = df_base[df_base["agreement"] == "Disagree"]

        if not df_agree.empty:
            n_agree = df_agree["n_samples"].iloc[0]
            n_pos_agree = df_agree["total_positives"].iloc[0]
            prev_agree = df_agree["prevalence"].iloc[0]
            print(f"  AGREE:    {n_agree} samples, {n_pos_agree} positives (prevalence: {prev_agree:.3f})")

        if not df_disagree.empty:
            n_disagree = df_disagree["n_samples"].iloc[0]
            n_pos_disagree = df_disagree["total_positives"].iloc[0]
            prev_disagree = df_disagree["prevalence"].iloc[0]
            print(f"  DISAGREE: {n_disagree} samples, {n_pos_disagree} positives (prevalence: {prev_disagree:.3f})")

        # AP comparison per method
        print(f"\n  --- Average Precision (AP) ---")
        print(f"  {'METHOD':<22} | {'AP Agree':<10} | {'AP Disagree':<12} | {'Δ (Agree-Disagree)':<18} | {'Δ vs Random Agree':<18} | {'Δ vs Random Disagree':<20}")
        print("  " + "-"*120)

        methods = df_base["method"].unique()

        # Compute prevalences (random baseline)
        random_ap_agree = prev_agree if not df_agree.empty else 0
        random_ap_disagree = prev_disagree if not df_disagree.empty else 0

        for method in methods:
            df_m_agree = df_agree[df_agree["method"] == method]
            df_m_disagree = df_disagree[df_disagree["method"] == method]

            ap_agree = df_m_agree["AP"].iloc[0] if not df_m_agree.empty else np.nan
            ap_disagree = df_m_disagree["AP"].iloc[0] if not df_m_disagree.empty else np.nan

            if pd.notna(ap_agree) and pd.notna(ap_disagree):
                delta = ap_agree - ap_disagree
                delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
                delta_rand_agree = ap_agree - random_ap_agree
                delta_rand_disagree = ap_disagree - random_ap_disagree
                delta_rand_agree_str = f"+{delta_rand_agree:.4f}" if delta_rand_agree >= 0 else f"{delta_rand_agree:.4f}"
                delta_rand_disagree_str = f"+{delta_rand_disagree:.4f}" if delta_rand_disagree >= 0 else f"{delta_rand_disagree:.4f}"
            else:
                delta_str = "-"
                delta_rand_agree_str = "-"
                delta_rand_disagree_str = "-"

            ap_agree_str = f"{ap_agree:.4f}" if pd.notna(ap_agree) else "-"
            ap_disagree_str = f"{ap_disagree:.4f}" if pd.notna(ap_disagree) else "-"

            print(f"  {method:<22} | {ap_agree_str:<10} | {ap_disagree_str:<12} | {delta_str:<18} | {delta_rand_agree_str:<18} | {delta_rand_disagree_str:<20}")

        # P@k table for a few important k values
        print(f"\n  --- P@k Comparison (Agree vs Disagree) ---")
        key_methods = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2", "CrossEncoder"]
        key_methods = [m for m in key_methods if m in methods]

        for method in key_methods:
            print(f"\n  {method}:")
            print(f"    {'k':<6} | {'P@k Agree (yes/tot)':<22} | {'P@k Disagree (yes/tot)':<24} | {'Δ P@k':<10}")
            print("    " + "-"*75)

            for k in [10, 50, 100, 200, 300, 500]:
                if k not in config.k_values:
                    continue

                df_m_agree_k = df_agree[(df_agree["method"] == method) & (df_agree["k"] == k)]
                df_m_disagree_k = df_disagree[(df_disagree["method"] == method) & (df_disagree["k"] == k)]

                if not df_m_agree_k.empty and not df_m_disagree_k.empty:
                    p_agree = df_m_agree_k["P@k"].values[0]
                    p_disagree = df_m_disagree_k["P@k"].values[0]
                    c_agree = df_m_agree_k["count"].values[0]
                    c_disagree = df_m_disagree_k["count"].values[0]
                    t_agree = df_m_agree_k["total_positives"].values[0]
                    t_disagree = df_m_disagree_k["total_positives"].values[0]
                    delta_p = p_agree - p_disagree
                    delta_str = f"+{delta_p:.3f}" if delta_p >= 0 else f"{delta_p:.3f}"

                    print(f"    {k:<6} | {p_agree:.3f} ({c_agree:>3}/{t_agree:<3})         | {p_disagree:.3f} ({c_disagree:>3}/{t_disagree:<3})           | {delta_str}")
                else:
                    print(f"    {k:<6} | -                      | -                        | -")


def plot_agree_disagree_comparison(df_agree_disagree: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot comparing performance on agree vs disagree."""
    if df_agree_disagree.empty:
        return

    # Focus on Gold
    df_gold = df_agree_disagree[df_agree_disagree["base_target"] == "Gold"]
    if df_gold.empty:
        return

    methods_to_plot = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2",
                       "LLM_Union", "CrossEncoder", "BM25", "TF-IDF"]
    methods_to_plot = [m for m in methods_to_plot if m in df_gold["method"].unique()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter4': 'brown', 'LLM_Inter3': 'purple',
        'LLM_Inter2': 'red', 'LLM_Union': 'orange', 'CrossEncoder': 'green',
        'BM25': 'cyan', 'TF-IDF': 'blue'
    }

    # Plot P@k for Agree
    ax = axes[0]
    df_agree = df_gold[df_gold["agreement"] == "Agree"]
    for method in methods_to_plot:
        df_m = df_agree[df_agree["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["P@k"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    # Add random baseline (prevalence)
    if not df_agree.empty:
        prev = df_agree["prevalence"].iloc[0]
        ax.axhline(y=prev, color='black', linestyle='--', linewidth=1,
                  label=f'Random ({prev:.3f})', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("P@k")
    ax.set_title("Gold - AGREE cases (annotators agree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.0)

    # Plot P@k for Disagree
    ax = axes[1]
    df_disagree = df_gold[df_gold["agreement"] == "Disagree"]
    for method in methods_to_plot:
        df_m = df_disagree[df_disagree["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["P@k"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    # Add random baseline (prevalence)
    if not df_disagree.empty:
        prev = df_disagree["prevalence"].iloc[0]
        ax.axhline(y=prev, color='black', linestyle='--', linewidth=1,
                  label=f'Random ({prev:.3f})', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("P@k")
    ax.set_title("Gold - DISAGREE cases (annotators disagree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_agree_vs_disagree_delta(df_agree_disagree: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot showing the performance delta between agree and disagree."""
    if df_agree_disagree.empty:
        return

    df_gold = df_agree_disagree[df_agree_disagree["base_target"] == "Gold"]
    if df_gold.empty:
        return

    methods_to_plot = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2",
                       "LLM_Union", "CrossEncoder", "BM25", "TF-IDF"]
    methods_to_plot = [m for m in methods_to_plot if m in df_gold["method"].unique()]

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter4': 'brown', 'LLM_Inter3': 'purple',
        'LLM_Inter2': 'red', 'LLM_Union': 'orange', 'CrossEncoder': 'green',
        'BM25': 'cyan', 'TF-IDF': 'blue'
    }

    df_agree = df_gold[df_gold["agreement"] == "Agree"]
    df_disagree = df_gold[df_gold["agreement"] == "Disagree"]

    for method in methods_to_plot:
        df_m_agree = df_agree[df_agree["method"] == method].sort_values("k")
        df_m_disagree = df_disagree[df_disagree["method"] == method].sort_values("k")

        if not df_m_agree.empty and not df_m_disagree.empty:
            # Merge on k
            merged = df_m_agree[["k", "P@k"]].merge(
                df_m_disagree[["k", "P@k"]], on="k", suffixes=("_agree", "_disagree")
            )
            merged["delta"] = merged["P@k_agree"] - merged["P@k_disagree"]

            ax.plot(merged["k"], merged["delta"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel("k")
    ax.set_ylabel("Δ P@k (Agree - Disagree)")
    ax.set_title("Gold - P@k difference between AGREE and DISAGREE cases\n(positive = better on Agree, negative = better on Disagree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def print_false_positive_analysis(df_fp: pd.DataFrame, config: Config):
    """Print the false-positive analysis."""
    print("\n" + "="*180)
    print("FALSE-POSITIVE ANALYSIS - Do the methods limit FP?")
    print("="*180)

    if df_fp.empty:
        print("  No data available.")
        return

    target = "Gold"
    df_t = df_fp[df_fp["target"] == target]
    if df_t.empty:
        print(f"  No data for target={target}")
        return

    # Basic stats
    n_pos = df_t["n_positives"].iloc[0]
    n_neg = df_t["n_negatives"].iloc[0]
    n_total = n_pos + n_neg

    print(f"\n  TARGET: {target}")
    print(f"  Total: {n_total} samples | {n_pos} positives (YES) | {n_neg} negatives (NO)")
    print(f"  Prevalence: {n_pos/n_total:.1%}" if n_total > 0 else "  Prevalence: N/A")

    # TP/FP comparison table
    print(f"\n  --- True Positives (TP) and False Positives (FP) at each k ---")

    methods = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2", "LLM_Union", "CrossEncoder"]
    methods = [m for m in methods if m in df_t["method"].unique()]

    print(f"\n  {'k':<6} | {'Random':<20}", end="")
    for method in methods:
        print(f" | {method:<20}", end="")
    print()

    print(f"  {'':6} | {'TP/FP (FP%)':<20}", end="")
    for _ in methods:
        print(f" | {'TP/FP (FP%)':<20}", end="")
    print()
    print("  " + "-"*(7 + 23 + 23*len(methods)))

    for k in [10, 50, 100, 200, 300, 500]:
        if k not in config.k_values:
            continue

        row = f"  {k:<6}"

        # Random baseline (computed from n_pos, n_neg)
        prevalence = n_pos / n_total
        rand_tp = k * prevalence
        rand_fp = k - rand_tp
        rand_fp_pct = rand_fp / k * 100
        row += f" | {rand_tp:>5.0f}/{rand_fp:<5.0f} ({rand_fp_pct:>4.1f}%)"

        for method in methods:
            df_m = df_t[(df_t["method"] == method) & (df_t["k"] == k)]
            if not df_m.empty:
                tp = df_m["TP"].values[0]
                fp = df_m["FP"].values[0]
                fp_pct = df_m["FP_Rate"].values[0] * 100
                row += f" | {tp:>5}/{fp:<5} ({fp_pct:>4.1f}%)"
            else:
                row += " | -                   "
        print(row)

    # Analysis of FP origin (Agree vs Disagree)
    print(f"\n  --- False-Positive Origin: Agree vs Disagree ---")
    print(f"  (FP on Disagree are cases where the model says YES but Gold says NO,")
    print(f"   while the human annotators already disagreed)")

    print(f"\n  {'k':<6}", end="")
    for method in methods:
        print(f" | {method:<28}", end="")
    print()

    print(f"  {'':6}", end="")
    for _ in methods:
        print(f" | {'FP_Agree / FP_Disagree':<28}", end="")
    print()
    print("  " + "-"*(7 + 31*len(methods)))

    for k in [50, 100, 200, 300, 500]:
        if k not in config.k_values:
            continue

        row = f"  {k:<6}"
        for method in methods:
            df_m = df_t[(df_t["method"] == method) & (df_t["k"] == k)]
            if not df_m.empty:
                fp = df_m["FP"].values[0]
                fp_agree = df_m["FP_Agree"].values[0]
                fp_disagree = df_m["FP_Disagree"].values[0]
                if fp > 0:
                    pct_disagree = fp_disagree / fp * 100
                    row += f" | {fp_agree:>4} / {fp_disagree:<4} ({pct_disagree:>4.1f}% Disagree)"
                else:
                    row += f" | 0 / 0 (N/A)              "
            else:
                row += " | -                           "
        print(row)

    # FP reduction vs Random
    print(f"\n  --- False-Positive Reduction vs Random ---")
    print(f"  (How many FP avoided compared to a random selection)")

    print(f"\n  {'k':<6}", end="")
    for method in methods:
        print(f" | {method:<18}", end="")
    print()

    print(f"  {'':6}", end="")
    for _ in methods:
        print(f" | {'FP avoided (%red)':<18}", end="")
    print()
    print("  " + "-"*(7 + 21*len(methods)))

    for k in [50, 100, 200, 300, 500]:
        if k not in config.k_values:
            continue

        row = f"  {k:<6}"
        rand_fp = k * (n_neg / n_total)
        for method in methods:
            df_m = df_t[(df_t["method"] == method) & (df_t["k"] == k)]
            if not df_m.empty:
                fp = df_m["FP"].values[0]
                fp_reduction = rand_fp - fp
                if rand_fp > 0:
                    pct_reduction = fp_reduction / rand_fp * 100
                    row += f" | {fp_reduction:>+5.0f} ({pct_reduction:>+5.1f}%)"
                else:
                    row += f" | {fp_reduction:>+5.0f} (N/A)     "
            else:
                row += " | -                 "
        print(row)

    # Focus on key methods
    print(f"\n  --- Summary: FP filtering effectiveness ---")

    for method in ["Heuristic", "LLM_Inter4", "LLM_Inter3"]:
        if method not in df_t["method"].unique():
            continue

        print(f"\n  {method}:")
        df_m = df_t[df_t["method"] == method]

        for k in [100, 300, 500]:
            if k not in config.k_values:
                continue
            df_k = df_m[df_m["k"] == k]
            if df_k.empty:
                continue

            tp = df_k["TP"].values[0]
            fp = df_k["FP"].values[0]
            fp_agree = df_k["FP_Agree"].values[0]
            fp_disagree = df_k["FP_Disagree"].values[0]
            rand_fp = k * (n_neg / n_total)
            fp_reduction = rand_fp - fp

            print(f"    k={k}: {tp} TP, {fp} FP (vs {rand_fp:.0f} FP random)")
            print(f"           FP reduction: {fp_reduction:+.0f} ({fp_reduction/rand_fp*100:+.1f}%)")
            if fp > 0:
                print(f"           FP origin: {fp_agree} from Agree, {fp_disagree} from Disagree ({fp_disagree/fp*100:.1f}% Disagree)")


def plot_false_positive_analysis(df_fp: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot of the false-positive analysis."""
    if df_fp.empty:
        return

    df_t = df_fp[df_fp["target"] == "Gold"]
    if df_t.empty:
        return

    methods = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2", "CrossEncoder"]
    methods = [m for m in methods if m in df_t["method"].unique()]

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter4': 'brown', 'LLM_Inter3': 'purple',
        'LLM_Inter2': 'red', 'LLM_Union': 'orange', 'CrossEncoder': 'green'
    }

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot 1: FP count vs k
    ax = axes[0]
    for method in methods:
        df_m = df_t[df_t["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    # Random baseline
    df_rand = df_t[df_t["method"] == methods[0]].sort_values("k")
    if not df_rand.empty:
        ax.plot(df_rand["k"], df_rand["random_FP"], color='black', linestyle='--',
               linewidth=1.5, label='Random', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("Number of False Positives")
    ax.set_title("FP vs k (Gold)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Plot 2: FP reduction vs random
    ax = axes[1]
    for method in methods:
        df_m = df_t[df_t["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP_reduction_vs_random"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel("k")
    ax.set_ylabel("FP avoided vs Random")
    ax.set_title("FP Reduction vs Random")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Plot 3: % of FP coming from Disagree
    ax = axes[2]
    for method in methods:
        df_m = df_t[df_t["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP_disagree_pct"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    # Reference line: proportion of Disagree in the dataset
    # Disagree = 339/1015 = 33.4%
    ax.axhline(y=33.4, color='black', linestyle='--', linewidth=1,
              label='Base rate (33.4%)', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("% FP coming from Disagree")
    ax.set_title("FP Origin: % Disagree")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# PLOTS
# ============================================================
def plot_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualization."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple',
        'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['Random', 'TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in main_methods:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric}")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_inter3_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualization of the 4 combinations of 3 LLMs."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black',
        'Inter3_LQM': 'blue',
        'Inter3_LQQ32': 'green',
        'Inter3_LMQ32': 'red',
        'Inter3_QMQ32': 'purple',
        'LLM_Inter3': 'orange',
        'LLM_Inter4': 'brown'
    }

    methods_to_plot = ['Random', 'Inter3_LQM', 'Inter3_LQQ32', 'Inter3_LMQ32',
                       'Inter3_QMQ32', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in methods_to_plot:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric} (Inter3 Combinations)")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_coverage_curves(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Coverage curves."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        df_first = next(iter(coverage_dict.values()))
        df_t_first = df_first[df_first["target"] == target].sort_values("k")

        if len(df_t_first) > 0:
            ax.plot(df_t_first["k"], df_t_first["random_expected"],
                   color='black', linestyle='--', linewidth=1.5,
                   label='Random (expected)', alpha=0.7)

            # Horizontal line for the total positives
            total_pos = df_t_first["total_positives"].iloc[0]
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        for method in main_methods:
            if method not in coverage_dict:
                continue
            df_cov = coverage_dict[method]
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["positives_retrieved"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k")
        ax.set_ylabel("YES retrieved")
        ax.set_title(f"{target} - Num YES retrieved")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_gain_vs_random(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of the relative gain versus chance."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

        for method in main_methods:
            if method not in coverage_dict:
                continue
            df_cov = coverage_dict[method]
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["delta_vs_random"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k (number of documents)")
        ax.set_ylabel("Gain vs Random (number of YES)")
        ax.set_title(f"{target} - Gain versus chance")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_count_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of the number of YES retrieved per method."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple',
        'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['Random', 'TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        total_pos = None

        for method in main_methods:
            if method not in results_dict:
                continue
            df_res = results_dict[method]
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["count"],
                       color=colors.get(method, 'gray'),
                       marker='o' if method != 'Random' else '.',
                       linewidth=2 if method != 'Random' else 1,
                       linestyle='--' if method == 'Random' else '-',
                       label=method, alpha=0.7 if method == 'Random' else 1.0)

                if method != 'Random' and total_pos is None:
                    total_pos = df_t["total_positives"].iloc[0]

        # Add total line
        if total_pos is not None:
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        ax.set_xlabel("k")
        ax.set_ylabel("Number of YES retrieved")
        ax.set_title(f"{target} - YES retrieved vs k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# MAIN
# ============================================================
def main(filter_union: bool = False, compute_crossencoder: bool = True):
    """Main pipeline."""
    print("="*70)
    print("FULLY UNSUPERVISED RANKING PIPELINE (4 LLMs)")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print("="*70)
    print("""
4 LLMs: Llama, Qwen, Mistral, Qwen32B

COMBINATIONS TESTED:
---------------------
- Union: at least 1 LLM says yes
- Inter2: at least 2 LLMs say yes
- Inter3: at least 3 of the 4 LLMs say yes
- Inter4: all 4 LLMs say yes

Specific combinations of 3:
- Inter3_LQM: Llama + Qwen + Mistral
- Inter3_LQQ32: Llama + Qwen + Qwen32B
- Inter3_LMQ32: Llama + Mistral + Qwen32B
- Inter3_QMQ32: Qwen + Mistral + Qwen32B

TIE-BREAK HANDLING:
------------------------------------
The binary methods (LLM_Union, LLM_Inter2, LLM_Inter3, LLM_Inter4)
produce 0/1 scores creating ties. To break them
deterministically:

  Score_final = Score_LLM + ε × Score_CrossEncoder  (ε = 1e-6)

Documents with the same LLM vote are ordered by their CrossEncoder score.
The continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic) do not have
significant ties and use direct sorting.
""")

    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_unsupervised_4llm{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # Load & prepare
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # LLM intersection analysis
    llm_stats = analyze_llm_intersection_coverage(df)
    print_llm_intersection_analysis(llm_stats)

    # Compute all scores
    print("\n" + "="*70)
    print("COMPUTING RANKINGS")
    print("="*70)

    all_scores = compute_all_ranking_scores(df)

    results_dict = {}
    coverage_dict = {}

    n_methods = len(all_scores) + 1  # +1 for Random
    print(f"\n  [1/{n_methods}] Random baseline...")
    results_dict["Random"] = compute_random_baseline(df, config)

    for i, (method_name, scores) in enumerate(all_scores.items(), 2):
        print(f"  [{i}/{n_methods}] {method_name}...")
        results_dict[method_name] = evaluate_method(df, scores, method_name, config)
        coverage_dict[method_name] = analyze_positive_coverage(df, scores, method_name, config)

    # Reporting
    print_comparison_table(results_dict, config)
    print_coverage_table(coverage_dict, config)
    print_detailed_count_table(results_dict, config)

    # Agree vs Disagree analysis
    print("\n" + "="*70)
    print("AGREE vs DISAGREE ANALYSIS")
    print("="*70)
    df_agree_disagree = analyze_agree_disagree_performance(df, all_scores, config)
    print_agree_disagree_comparison(df_agree_disagree, config)

    # False-positive analysis
    print("\n" + "="*70)
    print("FALSE-POSITIVE ANALYSIS")
    print("="*70)
    df_fp = analyze_false_positives(df, all_scores, config)
    print_false_positive_analysis(df_fp, config)

    # False-positive analysis
    print("\n" + "="*70)
    print("FALSE-POSITIVE ANALYSIS")
    print("="*70)
    df_fp = analyze_false_positives(df, all_scores, config)
    print_false_positive_analysis(df_fp, config)

    # Plots
    plot_comparison(results_dict, config, f"{config.output_dir}/comparison_main.png")
    plot_inter3_comparison(results_dict, config, f"{config.output_dir}/comparison_inter3.png")
    plot_coverage_curves(coverage_dict, config, f"{config.output_dir}/coverage.png")
    plot_gain_vs_random(coverage_dict, config, f"{config.output_dir}/gain_vs_random.png")
    plot_count_comparison(results_dict, config, f"{config.output_dir}/count_comparison.png")

    # Plots Agree vs Disagree
    plot_agree_disagree_comparison(df_agree_disagree, config, f"{config.output_dir}/agree_vs_disagree.png")
    plot_agree_vs_disagree_delta(df_agree_disagree, config, f"{config.output_dir}/agree_disagree_delta.png")

    # False-positive plot
    plot_false_positive_analysis(df_fp, config, f"{config.output_dir}/false_positives.png")

    # Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    for name, scores in all_scores.items():
        df[f"score_{name.lower().replace('-', '_')}"] = scores

    df.to_csv(f"{config.output_dir}/df_with_scores_{timestamp}.csv", index=False)

    df_results = pd.concat(list(results_dict.values()), ignore_index=True)
    df_results.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    df_coverage = pd.concat(list(coverage_dict.values()), ignore_index=True)
    df_coverage.to_csv(f"{config.output_dir}/coverage_{timestamp}.csv", index=False)

    # Save agree/disagree results
    df_agree_disagree.to_csv(f"{config.output_dir}/agree_disagree_results_{timestamp}.csv", index=False)

    # Save FP results
    df_fp.to_csv(f"{config.output_dir}/false_positives_{timestamp}.csv", index=False)

    with open(f"{config.output_dir}/llm_stats_{timestamp}.json", "w") as f:
        json.dump(llm_stats, f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict, coverage_dict, llm_stats, df_agree_disagree, df_fp


if __name__ == "__main__":
    df, results, coverage, llm_stats, agree_disagree, fp_analysis = main(filter_union=False, compute_crossencoder=True)

# 5 LLMs

In [ ]:
# ============================================================
# FULLY UNSUPERVISED PIPELINE - NO LABELS USED
# 5 LLMs VERSION: Llama, Qwen, Mistral, Qwen32B, Qwen3_32B_Thinking
# ============================================================
"""
Fully unsupervised pipeline for legal ranking.

TIE-BREAK HANDLING:
====================================
The binary methods (LLM_Union, LLM_Inter*, etc.) produce
binary scores (0 or 1), creating many ties. To break them:

  Score final = Score_LLM + ε × Score_CrossEncoder

where ε = 1e-6, small enough for the LLM score to dominate.

COMBINATIONS TESTED (5 LLMs):
==============================
- Union: at least 1 LLM says yes
- Inter2: at least 2 LLMs say yes
- Inter3: at least 3 LLMs say yes
- Inter4: at least 4 LLMs say yes
- Inter5: all 5 LLMs say yes

Specific combinations of 3 LLMs (C(5,3) = 10 combinations)
Specific combinations of 4 LLMs (C(5,4) = 5 combinations)

Reference: Voorhees (2000), TREC evaluation methodology
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, field
import json
from itertools import combinations

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# DEPENDENCY INSTALLATION
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess
    packages = [
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]
    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    qwen32b_predictions: str = ""
    qwen3_32b_thinking_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    # Targets split by agreement/disagreement
    targets_by_agreement: List[str] = field(default_factory=lambda: [
        "Gold_Agree", "Gold_Disagree",
        "A1_Agree", "A1_Disagree",
        "A2_Agree", "A2_Disagree"
    ])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    random_seed: int = 42

    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    # Names of the 5 LLMs for reference
    llm_names: List[str] = field(default_factory=lambda: ["Llama", "Qwen", "Mistral", "Qwen32B", "Qwen3Think"])

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.qwen32b_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-32b.xlsx"
        self.qwen3_32b_thinking_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen3-32b.xlsx"
        self.output_dir = f"artifacts/outputs_unsupervised_5llm"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        dcg = float(np.sum(y_sorted / np.log2(np.arange(2, k + 2))))

        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:k]
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k + 2))))

        return dcg / idcg if idcg > 0 else 0.0

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0


# ============================================================
# DATA LOADING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 5 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA (5 LLMs)")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)
    df_qwen32b = pd.read_excel(config.qwen32b_predictions)
    df_qwen3_think = pd.read_excel(config.qwen3_32b_thinking_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)} | Qwen32B: {len(df_qwen32b)} | Qwen3Think: {len(df_qwen3_think)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]
    for df in [df_main, df_llama, df_qwen, df_mistral, df_qwen32b, df_qwen3_think]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    # Llama
    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    # Qwen (7B)
    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    # Mistral
    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    # Qwen 32B
    pred_col_q32 = None
    for col in df_qwen32b.columns:
        if "pred" in col.lower() and col not in key_cols:
            pred_col_q32 = col
            break

    if pred_col_q32 is None:
        for col in df_qwen32b.columns:
            if col not in key_cols and col != "_key":
                vals = df_qwen32b[col].dropna().unique()
                if set(vals).issubset({0, 1, "0", "1", "oui", "non", "Oui", "Non"}):
                    pred_col_q32 = col
                    break

    if pred_col_q32:
        df_qwen32b = df_qwen32b.rename(columns={pred_col_q32: "Qwen32B_Pred"})
    else:
        df_qwen32b["Qwen32B_Pred"] = 0
        print("  WARNING: Could not find prediction column for Qwen32B, defaulting to 0")

    qwen32b_cols = df_qwen32b[["_key", "Qwen32B_Pred"]].copy()

    # Qwen3 32B Thinking - find the prediction column
    pred_col_q3think = None
    # First look for explicit columns
    for col in df_qwen3_think.columns:
        col_lower = col.lower()
        if "pred" in col_lower and col not in key_cols and col != "_key":
            pred_col_q3think = col
            break

    if pred_col_q3think is None:
        # Look for a column with binary values
        for col in df_qwen3_think.columns:
            if col not in key_cols and col != "_key" and "thinking" not in col.lower():
                vals = df_qwen3_think[col].dropna().unique()
                if len(vals) <= 3 and set(vals).issubset({0, 1, "0", "1", "oui", "non", "Oui", "Non"}):
                    pred_col_q3think = col
                    break

    if pred_col_q3think:
        df_qwen3_think = df_qwen3_think.rename(columns={pred_col_q3think: "Qwen3Think_Pred"})
        print(f"  Qwen3Think prediction column found: {pred_col_q3think}")
    else:
        df_qwen3_think["Qwen3Think_Pred"] = 0
        print("  WARNING: Could not find prediction column for Qwen3Think, defaulting to 0")

    qwen3_think_cols = df_qwen3_think[["_key", "Qwen3Think_Pred"]].copy()

    # Merge all
    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")
    df = df.merge(qwen32b_cols, on="_key", how="left")
    df = df.merge(qwen3_think_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)

    # Build labels for agree/disagree cases
    df["is_agree"] = (df["lbl_A2"] == df["lbl_A1"]) & df["lbl_A2"].notna() & df["lbl_A1"].notna()
    df["is_disagree"] = (df["lbl_A2"] != df["lbl_A1"]) & df["lbl_A2"].notna() & df["lbl_A1"].notna()

    # Labels filtered by agreement/disagreement
    df["lbl_Gold_Agree"] = df.apply(lambda r: r["lbl_Gold"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_Gold_Disagree"] = df.apply(lambda r: r["lbl_Gold"] if r["is_disagree"] else np.nan, axis=1)
    df["lbl_A1_Agree"] = df.apply(lambda r: r["lbl_A1"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_A1_Disagree"] = df.apply(lambda r: r["lbl_A1"] if r["is_disagree"] else np.nan, axis=1)
    df["lbl_A2_Agree"] = df.apply(lambda r: r["lbl_A2"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_A2_Disagree"] = df.apply(lambda r: r["lbl_A2"] if r["is_disagree"] else np.nan, axis=1)

    # Agree/disagree stats
    n_agree = df["is_agree"].sum()
    n_disagree = df["is_disagree"].sum()
    print(f"\n  Agreement/Disagreement between annotators:")
    print(f"    - Agree (A2 == A1):   {n_agree} ({n_agree/len(df)*100:.1f}%)")
    print(f"    - Disagree (A2 != A1): {n_disagree} ({n_disagree/len(df)*100:.1f}%)")

    df = df.drop(columns=["_key"])

    return df


# ============================================================
# FEATURES
# ============================================================
def compute_tfidf_similarity(df: pd.DataFrame) -> np.ndarray:
    """TF-IDF similarity."""
    stopwords_fr = [
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ]

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2,
        max_df=0.95, stop_words=stopwords_fr, lowercase=True
    )
    vectorizer.fit(texts + articles)

    tfidf_texts = vectorizer.transform(texts)
    tfidf_articles = vectorizer.transform(articles)

    return np.array([
        cosine_similarity(tfidf_texts[i], tfidf_articles[i])[0, 0]
        for i in range(len(texts))
    ])


def compute_bm25_scores(df: pd.DataFrame) -> np.ndarray:
    """BM25 scores."""
    from rank_bm25 import BM25Okapi

    stopwords = set(["le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "à"])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords and len(t) > 2]

    articles = df["article_text"].fillna("").tolist()
    texts = df["text"].fillna("").tolist()

    tokenized_articles = [tokenize(a) for a in articles]
    bm25 = BM25Okapi(tokenized_articles)

    scores = []
    for i, text in enumerate(texts):
        query = tokenize(text)
        if query:
            all_scores = bm25.get_scores(query)
            scores.append(all_scores[i])
        else:
            scores.append(0.0)

    scores = np.array(scores)
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores


def compute_cross_encoder_scores(df: pd.DataFrame, config: Config) -> np.ndarray:
    """Cross-encoder 0-shot."""
    print("\n  Computing Cross-Encoder scores...")

    from sentence_transformers import CrossEncoder
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"    Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()
    pairs = [[t, a] for t, a in zip(texts, articles)]

    scores = []
    batch_size = config.cross_encoder_batch_size

    for i in tqdm(range(0, len(pairs), batch_size), desc="    Cross-encoder"):
        batch = pairs[i:i + batch_size]
        batch_scores = model.predict(batch, show_progress_bar=False)
        scores.extend(batch_scores)

    scores = np.array(scores)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    print(f"    Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    return scores_normalized


def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all UNSUPERVISED features with 5 LLMs."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION (UNSUPERVISED - 5 LLMs)")
    print("="*70)

    df = df.copy()

    # Individual votes of the 5 LLMs
    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)
    df["vote_qwen32b"] = (df["Qwen32B_Pred"] == 1).astype(int)
    df["vote_qwen3think"] = (df["Qwen3Think_Pred"] == 1).astype(int)

    # Sum of votes (0-5)
    df["vote_sum"] = (df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"] +
                      df["vote_qwen32b"] + df["vote_qwen3think"])

    # Union: at least 1 LLM says yes
    df["vote_union"] = (df["vote_sum"] >= 1).astype(int)

    # Inter2: at least 2 LLMs say yes
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)

    # Inter3: at least 3 LLMs say yes
    df["vote_inter3"] = (df["vote_sum"] >= 3).astype(int)

    # Inter4: at least 4 LLMs say yes
    df["vote_inter4"] = (df["vote_sum"] >= 4).astype(int)

    # Inter5: all 5 LLMs say yes
    df["vote_inter5"] = (df["vote_sum"] == 5).astype(int)

    # ============================================================
    # Specific combinations of 4 LLMs (C(5,4) = 5 combinations)
    # ============================================================
    # L = Llama, Q = Qwen, M = Mistral, Q32 = Qwen32B, Q3T = Qwen3Think
    df["vote_inter4_LQMQ32"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) &
                                 (df["vote_mistral"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter4_LQMQ3T"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) &
                                 (df["vote_mistral"] == 1) & (df["vote_qwen3think"] == 1)).astype(int)
    df["vote_inter4_LQQ32Q3T"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) &
                                   (df["vote_qwen32b"] == 1) & (df["vote_qwen3think"] == 1)).astype(int)
    df["vote_inter4_LMQ32Q3T"] = ((df["vote_llama"] == 1) & (df["vote_mistral"] == 1) &
                                   (df["vote_qwen32b"] == 1) & (df["vote_qwen3think"] == 1)).astype(int)
    df["vote_inter4_QMQ32Q3T"] = ((df["vote_qwen"] == 1) & (df["vote_mistral"] == 1) &
                                   (df["vote_qwen32b"] == 1) & (df["vote_qwen3think"] == 1)).astype(int)

    # ============================================================
    # Specific combinations of 3 LLMs (C(5,3) = 10 combinations)
    # ============================================================
    df["vote_inter3_LQM"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) &
                              (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter3_LQQ32"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) &
                                (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_LQQ3T"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) &
                                (df["vote_qwen3think"] == 1)).astype(int)
    df["vote_inter3_LMQ32"] = ((df["vote_llama"] == 1) & (df["vote_mistral"] == 1) &
                                (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_LMQ3T"] = ((df["vote_llama"] == 1) & (df["vote_mistral"] == 1) &
                                (df["vote_qwen3think"] == 1)).astype(int)
    df["vote_inter3_LQ32Q3T"] = ((df["vote_llama"] == 1) & (df["vote_qwen32b"] == 1) &
                                  (df["vote_qwen3think"] == 1)).astype(int)
    df["vote_inter3_QMQ32"] = ((df["vote_qwen"] == 1) & (df["vote_mistral"] == 1) &
                                (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_QMQ3T"] = ((df["vote_qwen"] == 1) & (df["vote_mistral"] == 1) &
                                (df["vote_qwen3think"] == 1)).astype(int)
    df["vote_inter3_QQ32Q3T"] = ((df["vote_qwen"] == 1) & (df["vote_qwen32b"] == 1) &
                                  (df["vote_qwen3think"] == 1)).astype(int)
    df["vote_inter3_MQ32Q3T"] = ((df["vote_mistral"] == 1) & (df["vote_qwen32b"] == 1) &
                                  (df["vote_qwen3think"] == 1)).astype(int)

    print("  [1/3] TF-IDF...")
    df["sim_tfidf"] = compute_tfidf_similarity(df)

    print("  [2/3] BM25...")
    df["sim_bm25"] = compute_bm25_scores(df)

    if compute_crossencoder:
        print("  [3/3] Cross-Encoder...")
        df["sim_crossencoder"] = compute_cross_encoder_scores(df, config)
    else:
        df["sim_crossencoder"] = 0.0

    return df


# ============================================================
# RANKING SCORES (with integrated tie-break)
# ============================================================
def compute_all_ranking_scores(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    """
    Compute ranking scores for all methods.

    TIE-BREAK HANDLING:
    - Continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic): direct scores
    - Binary methods (LLM_*): score = vote + ε × CrossEncoder to break ties
    """
    scores = {}

    # Continuous methods - no tie-break needed
    scores["TF-IDF"] = df["sim_tfidf"].values
    scores["BM25"] = df["sim_bm25"].values
    scores["CrossEncoder"] = df["sim_crossencoder"].values

    # Heuristic - weighted combination (near-unique scores) - updated for 5 LLMs
    weights = {
        "vote_llama": 0.20, "vote_qwen": 0.20, "vote_mistral": 0.20,
        "vote_qwen32b": 0.20, "vote_qwen3think": 0.20,
        "vote_inter2": 0.2, "vote_inter3": 0.4, "vote_inter4": 0.7, "vote_inter5": 1.0,
        "sim_tfidf": 0.15, "sim_bm25": 0.15, "sim_crossencoder": 0.35,
    }
    heuristic = np.zeros(len(df))
    for feature, weight in weights.items():
        if feature in df.columns:
            heuristic += weight * df[feature].values
    scores["Heuristic"] = heuristic

    # Binary methods - tie-break via CrossEncoder
    eps = 1e-6
    ce_norm = df["sim_crossencoder"].values
    if ce_norm.max() > ce_norm.min():
        ce_norm = (ce_norm - ce_norm.min()) / (ce_norm.max() - ce_norm.min())

    # Main methods
    scores["LLM_Union"] = df["vote_union"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter2"] = df["vote_inter2"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter3"] = df["vote_inter3"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter4"] = df["vote_inter4"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter5"] = df["vote_inter5"].values.astype(float) + eps * ce_norm

    # The 5 specific combinations of 4 LLMs
    scores["Inter4_LQMQ32"] = df["vote_inter4_LQMQ32"].values.astype(float) + eps * ce_norm
    scores["Inter4_LQMQ3T"] = df["vote_inter4_LQMQ3T"].values.astype(float) + eps * ce_norm
    scores["Inter4_LQQ32Q3T"] = df["vote_inter4_LQQ32Q3T"].values.astype(float) + eps * ce_norm
    scores["Inter4_LMQ32Q3T"] = df["vote_inter4_LMQ32Q3T"].values.astype(float) + eps * ce_norm
    scores["Inter4_QMQ32Q3T"] = df["vote_inter4_QMQ32Q3T"].values.astype(float) + eps * ce_norm

    # The 10 specific combinations of 3 LLMs
    scores["Inter3_LQM"] = df["vote_inter3_LQM"].values.astype(float) + eps * ce_norm
    scores["Inter3_LQQ32"] = df["vote_inter3_LQQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_LQQ3T"] = df["vote_inter3_LQQ3T"].values.astype(float) + eps * ce_norm
    scores["Inter3_LMQ32"] = df["vote_inter3_LMQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_LMQ3T"] = df["vote_inter3_LMQ3T"].values.astype(float) + eps * ce_norm
    scores["Inter3_LQ32Q3T"] = df["vote_inter3_LQ32Q3T"].values.astype(float) + eps * ce_norm
    scores["Inter3_QMQ32"] = df["vote_inter3_QMQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_QMQ3T"] = df["vote_inter3_QMQ3T"].values.astype(float) + eps * ce_norm
    scores["Inter3_QQ32Q3T"] = df["vote_inter3_QQ32Q3T"].values.astype(float) + eps * ce_norm
    scores["Inter3_MQ32Q3T"] = df["vote_inter3_MQ32Q3T"].values.astype(float) + eps * ce_norm

    return scores


# ============================================================
# EVALUATION (with count and ratio)
# ============================================================
def evaluate_method(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate a ranking with the number of positives retrieved."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            count_k = Metrics.count_at_k(y, s, k)
            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count": count_k,
                "total_positives": n_pos,
                "count_ratio": f"{count_k}/{n_pos}",
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """Random baseline = prevalence."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        for k in config.k_values:
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,
                "count": int(expected_pos_at_k),
                "total_positives": n_pos,
                "count_ratio": f"{int(expected_pos_at_k)}/{n_pos}",
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# COVERAGE ANALYSIS
# ============================================================
def analyze_positive_coverage(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Analysis: how many positives retrieved at each k."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))

        if n_pos == 0:
            continue

        prevalence = n_pos / n_samples
        order = np.argsort(s)[::-1]
        y_sorted = y[order]

        for k in config.k_values:
            k_actual = min(k, n_samples)
            positives_at_k = int(np.sum(y_sorted[:k_actual]))
            random_expected = k_actual * prevalence

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "positives_retrieved": positives_at_k,
                "total_positives": n_pos,
                "coverage": positives_at_k / n_pos,
                "count_ratio": f"{positives_at_k}/{n_pos}",
                "random_expected": random_expected,
                "delta_vs_random": positives_at_k - random_expected,
                "n_samples": n_samples,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def analyze_llm_intersection_coverage(df: pd.DataFrame) -> Dict:
    """Coverage analysis by LLM intersections (5 LLMs)."""
    stats = {
        "inter5": int((df["vote_inter5"] == 1).sum()),
        "inter4": int((df["vote_inter4"] == 1).sum()),
        "inter3": int((df["vote_inter3"] == 1).sum()),
        "inter2": int((df["vote_inter2"] == 1).sum()),
        "union": int((df["vote_union"] == 1).sum()),
        "no_vote": int((df["vote_union"] == 0).sum()),
        "total": int(len(df))
    }

    # Combinations of 4 LLMs
    stats["inter4_LQMQ32"] = int((df["vote_inter4_LQMQ32"] == 1).sum())
    stats["inter4_LQMQ3T"] = int((df["vote_inter4_LQMQ3T"] == 1).sum())
    stats["inter4_LQQ32Q3T"] = int((df["vote_inter4_LQQ32Q3T"] == 1).sum())
    stats["inter4_LMQ32Q3T"] = int((df["vote_inter4_LMQ32Q3T"] == 1).sum())
    stats["inter4_QMQ32Q3T"] = int((df["vote_inter4_QMQ32Q3T"] == 1).sum())

    # Combinations of 3 LLMs
    stats["inter3_LQM"] = int((df["vote_inter3_LQM"] == 1).sum())
    stats["inter3_LQQ32"] = int((df["vote_inter3_LQQ32"] == 1).sum())
    stats["inter3_LQQ3T"] = int((df["vote_inter3_LQQ3T"] == 1).sum())
    stats["inter3_LMQ32"] = int((df["vote_inter3_LMQ32"] == 1).sum())
    stats["inter3_LMQ3T"] = int((df["vote_inter3_LMQ3T"] == 1).sum())
    stats["inter3_LQ32Q3T"] = int((df["vote_inter3_LQ32Q3T"] == 1).sum())
    stats["inter3_QMQ32"] = int((df["vote_inter3_QMQ32"] == 1).sum())
    stats["inter3_QMQ3T"] = int((df["vote_inter3_QMQ3T"] == 1).sum())
    stats["inter3_QQ32Q3T"] = int((df["vote_inter3_QQ32Q3T"] == 1).sum())
    stats["inter3_MQ32Q3T"] = int((df["vote_inter3_MQ32Q3T"] == 1).sum())

    # Per-LLM stats
    stats["llama_yes"] = int((df["vote_llama"] == 1).sum())
    stats["qwen_yes"] = int((df["vote_qwen"] == 1).sum())
    stats["mistral_yes"] = int((df["vote_mistral"] == 1).sum())
    stats["qwen32b_yes"] = int((df["vote_qwen32b"] == 1).sum())
    stats["qwen3think_yes"] = int((df["vote_qwen3think"] == 1).sum())

    for target in ["A1", "A2", "Gold"]:
        label_col = f"lbl_{target}"
        if label_col not in df.columns:
            continue

        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        df_target = df[mask].copy()
        y = df_target[label_col].astype(int)

        stats[f"{target}_total"] = int(len(y))
        stats[f"{target}_positives"] = int(y.sum())
        stats[f"{target}_pos_inter5"] = int(y[df_target["vote_inter5"] == 1].sum())
        stats[f"{target}_pos_inter4"] = int(y[df_target["vote_inter4"] == 1].sum())
        stats[f"{target}_pos_inter3"] = int(y[df_target["vote_inter3"] == 1].sum())
        stats[f"{target}_pos_inter2"] = int(y[df_target["vote_inter2"] == 1].sum())
        stats[f"{target}_pos_union"] = int(y[df_target["vote_union"] == 1].sum())
        stats[f"{target}_pos_no_vote"] = int(y[df_target["vote_union"] == 0].sum())

        # Combinations of 4 LLMs
        stats[f"{target}_pos_inter4_LQMQ32"] = int(y[df_target["vote_inter4_LQMQ32"] == 1].sum())
        stats[f"{target}_pos_inter4_LQMQ3T"] = int(y[df_target["vote_inter4_LQMQ3T"] == 1].sum())
        stats[f"{target}_pos_inter4_LQQ32Q3T"] = int(y[df_target["vote_inter4_LQQ32Q3T"] == 1].sum())
        stats[f"{target}_pos_inter4_LMQ32Q3T"] = int(y[df_target["vote_inter4_LMQ32Q3T"] == 1].sum())
        stats[f"{target}_pos_inter4_QMQ32Q3T"] = int(y[df_target["vote_inter4_QMQ32Q3T"] == 1].sum())

        # Combinations of 3 LLMs
        stats[f"{target}_pos_inter3_LQM"] = int(y[df_target["vote_inter3_LQM"] == 1].sum())
        stats[f"{target}_pos_inter3_LQQ32"] = int(y[df_target["vote_inter3_LQQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_LQQ3T"] = int(y[df_target["vote_inter3_LQQ3T"] == 1].sum())
        stats[f"{target}_pos_inter3_LMQ32"] = int(y[df_target["vote_inter3_LMQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_LMQ3T"] = int(y[df_target["vote_inter3_LMQ3T"] == 1].sum())
        stats[f"{target}_pos_inter3_LQ32Q3T"] = int(y[df_target["vote_inter3_LQ32Q3T"] == 1].sum())
        stats[f"{target}_pos_inter3_QMQ32"] = int(y[df_target["vote_inter3_QMQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_QMQ3T"] = int(y[df_target["vote_inter3_QMQ3T"] == 1].sum())
        stats[f"{target}_pos_inter3_QQ32Q3T"] = int(y[df_target["vote_inter3_QQ32Q3T"] == 1].sum())
        stats[f"{target}_pos_inter3_MQ32Q3T"] = int(y[df_target["vote_inter3_MQ32Q3T"] == 1].sum())

    return stats


def analyze_agree_disagree_performance(
    df: pd.DataFrame,
    all_scores: Dict[str, np.ndarray],
    config: Config
) -> pd.DataFrame:
    """Analyze performance on agree vs disagree cases."""
    results = []

    targets_agreement = ["Gold_Agree", "Gold_Disagree", "A1_Agree", "A1_Disagree",
                         "A2_Agree", "A2_Disagree"]

    for method_name, scores in all_scores.items():
        for target in targets_agreement:
            label_col = f"lbl_{target}"
            if label_col not in df.columns:
                continue

            mask = df[label_col].notna()
            if mask.sum() == 0:
                continue

            y = df.loc[mask, label_col].values.astype(int)
            s = scores[mask.values]

            n_samples = len(y)
            n_pos = int(np.sum(y))
            if n_samples == 0:
                continue

            prevalence = n_pos / n_samples
            ap = Metrics.average_precision(y, s)

            parts = target.rsplit('_', 1)
            base_target = parts[0]
            agreement_type = parts[1] if len(parts) > 1 else "All"

            for k in config.k_values:
                count_k = Metrics.count_at_k(y, s, k)
                results.append({
                    "method": method_name,
                    "target": target,
                    "base_target": base_target,
                    "agreement": agreement_type,
                    "k": k,
                    "P@k": Metrics.precision_at_k(y, s, k),
                    "R@k": Metrics.recall_at_k(y, s, k),
                    "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                    "count": count_k,
                    "total_positives": n_pos,
                    "count_ratio": f"{count_k}/{n_pos}",
                    "AP": ap,
                    "n_samples": n_samples,
                    "prevalence": prevalence
                })

    return pd.DataFrame(results)


def analyze_false_positives(
    df: pd.DataFrame,
    all_scores: Dict[str, np.ndarray],
    config: Config
) -> pd.DataFrame:
    """Detailed false-positive analysis for each method."""
    results = []

    for method_name, scores in all_scores.items():
        for target in config.targets:
            label_col = f"lbl_{target}"
            mask = df[label_col].notna()

            if mask.sum() == 0:
                continue

            df_subset = df[mask].copy()
            y = df_subset[label_col].values.astype(int)
            s = scores[mask.values]

            n_samples = len(y)
            n_pos = int(np.sum(y))
            n_neg = n_samples - n_pos

            order = np.argsort(s)[::-1]
            y_sorted = y[order]
            df_sorted = df_subset.iloc[order].copy()

            for k in config.k_values:
                k_actual = min(k, n_samples)

                y_topk = y_sorted[:k_actual]
                df_topk = df_sorted.iloc[:k_actual]

                tp = int(np.sum(y_topk == 1))
                fp = int(np.sum(y_topk == 0))

                precision = tp / k_actual if k_actual > 0 else 0
                recall = tp / n_pos if n_pos > 0 else 0

                fp_agree = 0
                fp_disagree = 0

                if "is_agree" in df_topk.columns and "is_disagree" in df_topk.columns:
                    fp_mask = (y_topk == 0)
                    fp_indices = df_topk.index[fp_mask]

                    for idx in fp_indices:
                        if df.loc[idx, "is_agree"]:
                            fp_agree += 1
                        elif df.loc[idx, "is_disagree"]:
                            fp_disagree += 1

                fp_rate = fp / k_actual if k_actual > 0 else 0
                fp_agree_rate = fp_agree / fp if fp > 0 else 0
                fp_disagree_rate = fp_disagree / fp if fp > 0 else 0

                random_fp = k_actual * (n_neg / n_samples)
                fp_reduction = (random_fp - fp) / random_fp if random_fp > 0 else 0

                results.append({
                    "method": method_name,
                    "target": target,
                    "k": k,
                    "TP": tp,
                    "FP": fp,
                    "FP_Agree": fp_agree,
                    "FP_Disagree": fp_disagree,
                    "Precision": precision,
                    "Recall": recall,
                    "FP_Rate": fp_rate,
                    "FP_Agree_Rate": fp_agree_rate,
                    "FP_Disagree_Rate": fp_disagree_rate,
                    "Random_FP_Expected": random_fp,
                    "FP_Reduction_vs_Random": fp_reduction,
                    "n_positives": n_pos,
                    "n_negatives": n_neg,
                    "n_samples": n_samples
                })

    return pd.DataFrame(results)


# ============================================================
# REPORTING (with count/total)
# ============================================================
def print_llm_intersection_analysis(stats: Dict):
    """Print the LLM intersection analysis (5 LLMs)."""
    print("\n" + "="*70)
    print("LLM INTERSECTION ANALYSIS (5 LLMs)")
    print("="*70)

    print(f"\n  Full dataset: {stats['total']} entries")
    print(f"\n  Individual votes:")
    print(f"    - Llama:      {stats['llama_yes']:>6} ({stats['llama_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen:       {stats['qwen_yes']:>6} ({stats['qwen_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Mistral:    {stats['mistral_yes']:>6} ({stats['mistral_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen32B:    {stats['qwen32b_yes']:>6} ({stats['qwen32b_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen3Think: {stats['qwen3think_yes']:>6} ({stats['qwen3think_yes']/stats['total']*100:>5.1f}%)")

    print(f"\n  Intersections:")
    print(f"    - 5 LLMs agree (inter5):  {stats['inter5']:>6} ({stats['inter5']/stats['total']*100:>5.1f}%)")
    print(f"    - 4+ LLMs agree (inter4): {stats['inter4']:>6} ({stats['inter4']/stats['total']*100:>5.1f}%)")
    print(f"    - 3+ LLMs agree (inter3): {stats['inter3']:>6} ({stats['inter3']/stats['total']*100:>5.1f}%)")
    print(f"    - 2+ LLMs agree (inter2): {stats['inter2']:>6} ({stats['inter2']/stats['total']*100:>5.1f}%)")
    print(f"    - 1+ LLM agrees (union):   {stats['union']:>6} ({stats['union']/stats['total']*100:>5.1f}%)")
    print(f"    - No LLM agrees:        {stats['no_vote']:>6} ({stats['no_vote']/stats['total']*100:>5.1f}%)")

    print(f"\n  Specific combinations of 4 LLMs:")
    print(f"    - L+Q+M+Q32 (LQMQ32):       {stats['inter4_LQMQ32']:>6} ({stats['inter4_LQMQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - L+Q+M+Q3T (LQMQ3T):       {stats['inter4_LQMQ3T']:>6} ({stats['inter4_LQMQ3T']/stats['total']*100:>5.1f}%)")
    print(f"    - L+Q+Q32+Q3T (LQQ32Q3T):   {stats['inter4_LQQ32Q3T']:>6} ({stats['inter4_LQQ32Q3T']/stats['total']*100:>5.1f}%)")
    print(f"    - L+M+Q32+Q3T (LMQ32Q3T):   {stats['inter4_LMQ32Q3T']:>6} ({stats['inter4_LMQ32Q3T']/stats['total']*100:>5.1f}%)")
    print(f"    - Q+M+Q32+Q3T (QMQ32Q3T):   {stats['inter4_QMQ32Q3T']:>6} ({stats['inter4_QMQ32Q3T']/stats['total']*100:>5.1f}%)")

    print(f"\n  Specific combinations of 3 LLMs:")
    print(f"    - L+Q+M (LQM):              {stats['inter3_LQM']:>6} ({stats['inter3_LQM']/stats['total']*100:>5.1f}%)")
    print(f"    - L+Q+Q32 (LQQ32):          {stats['inter3_LQQ32']:>6} ({stats['inter3_LQQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - L+Q+Q3T (LQQ3T):          {stats['inter3_LQQ3T']:>6} ({stats['inter3_LQQ3T']/stats['total']*100:>5.1f}%)")
    print(f"    - L+M+Q32 (LMQ32):          {stats['inter3_LMQ32']:>6} ({stats['inter3_LMQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - L+M+Q3T (LMQ3T):          {stats['inter3_LMQ3T']:>6} ({stats['inter3_LMQ3T']/stats['total']*100:>5.1f}%)")
    print(f"    - L+Q32+Q3T (LQ32Q3T):      {stats['inter3_LQ32Q3T']:>6} ({stats['inter3_LQ32Q3T']/stats['total']*100:>5.1f}%)")
    print(f"    - Q+M+Q32 (QMQ32):          {stats['inter3_QMQ32']:>6} ({stats['inter3_QMQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - Q+M+Q3T (QMQ3T):          {stats['inter3_QMQ3T']:>6} ({stats['inter3_QMQ3T']/stats['total']*100:>5.1f}%)")
    print(f"    - Q+Q32+Q3T (QQ32Q3T):      {stats['inter3_QQ32Q3T']:>6} ({stats['inter3_QQ32Q3T']/stats['total']*100:>5.1f}%)")
    print(f"    - M+Q32+Q3T (MQ32Q3T):      {stats['inter3_MQ32Q3T']:>6} ({stats['inter3_MQ32Q3T']/stats['total']*100:>5.1f}%)")

    for target in ["A1", "A2", "Gold"]:
        if f"{target}_total" not in stats:
            continue

        n_total = stats[f"{target}_total"]
        n_pos = stats[f"{target}_positives"]

        if n_pos == 0:
            continue

        print(f"\n  {target}: {n_pos} positives out of {n_total} ({n_pos/n_total*100:.1f}%)")
        print(f"    - In inter5:     {stats[f'{target}_pos_inter5']:>4}/{n_pos} ({stats[f'{target}_pos_inter5']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter4:     {stats[f'{target}_pos_inter4']:>4}/{n_pos} ({stats[f'{target}_pos_inter4']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter3:     {stats[f'{target}_pos_inter3']:>4}/{n_pos} ({stats[f'{target}_pos_inter3']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter2:     {stats[f'{target}_pos_inter2']:>4}/{n_pos} ({stats[f'{target}_pos_inter2']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In union:      {stats[f'{target}_pos_union']:>4}/{n_pos} ({stats[f'{target}_pos_union']/n_pos*100:>5.1f}% of positives)")
        print(f"    - Outside union:      {stats[f'{target}_pos_no_vote']:>4}/{n_pos} ({stats[f'{target}_pos_no_vote']/n_pos*100:>5.1f}% of positives)")


def print_comparison_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Comparison table with count/total."""
    print("\n" + "="*180)
    print("COMPARISON OF UNSUPERVISED METHODS (5 LLMs) - with num yes retrieved / num yes total")
    print("="*180)

    main_methods = ["Random", "TF-IDF", "BM25", "CrossEncoder", "Heuristic",
                   "LLM_Union", "LLM_Inter2", "LLM_Inter3", "LLM_Inter4", "LLM_Inter5"]
    main_methods = [m for m in main_methods if m in results_dict]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")
        print(f"{'='*180}")

        df_random = results_dict.get("Random")
        prevalence = None
        n_pos_total = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                n_pos_total = df_t_rand["total_positives"].iloc[0]
                print(f"  Prevalence (yes rate): {prevalence:.3f} ({prevalence*100:.1f}%)")
                print(f"  Total number of YES: {n_pos_total}")

        # Table AP
        print("\n  METHOD                | AP     | Δ vs Random")
        print("  " + "-"*50)
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = ap - prevalence
                    delta_str = f"+{delta:.3f}" if delta >= 0 else f"{delta:.3f}"
                else:
                    delta_str = "-"
                print(f"  {method:<22} | {ap:.4f} | {delta_str}")

        # P@k table for main methods
        print(f"\n  --- P@k for main methods (yes/tot) ---")
        print(f"  {'k':<6}", end="")
        for method in main_methods:
            print(f" | {method[:14]:<14}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in main_methods:
            print(f" | P@k   (yes/tot)", end="")
        print()
        print("  " + "-"*(7 + 18 * len(main_methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in main_methods:
                df_res = results_dict[method]
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    count = df_t["count"].values[0]
                    total = df_t["total_positives"].values[0]
                    row += f" | {p:.3f} ({count:>3}/{total:<3})"
                else:
                    row += " | -               "
            print(row)

        # P@k table for the 4-LLM combinations
        inter4_methods = ["Inter4_LQMQ32", "Inter4_LQMQ3T", "Inter4_LQQ32Q3T", "Inter4_LMQ32Q3T", "Inter4_QMQ32Q3T"]
        inter4_methods = [m for m in inter4_methods if m in results_dict]

        if inter4_methods:
            print(f"\n  --- P@k for combinations of 4 LLMs (yes/tot) ---")
            print(f"  {'k':<6}", end="")
            for method in inter4_methods:
                print(f" | {method:<18}", end="")
            print()
            print("  " + "-"*(7 + 21 * len(inter4_methods)))

            for k in [10, 50, 100, 200, 300, 500, 1000]:
                if k not in config.k_values:
                    continue
                row = f"  {k:<6}"
                for method in inter4_methods:
                    df_res = results_dict[method]
                    df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                    if len(df_t) > 0:
                        p = df_t["P@k"].values[0]
                        count = df_t["count"].values[0]
                        total = df_t["total_positives"].values[0]
                        row += f" | {p:.3f} ({count:>3}/{total:<3}) "
                    else:
                        row += " | -                  "
                print(row)

        # P@k table for the 3-LLM combinations
        inter3_methods = ["Inter3_LQM", "Inter3_LQQ32", "Inter3_LQQ3T", "Inter3_LMQ32", "Inter3_LMQ3T",
                          "Inter3_LQ32Q3T", "Inter3_QMQ32", "Inter3_QMQ3T", "Inter3_QQ32Q3T", "Inter3_MQ32Q3T"]
        inter3_methods = [m for m in inter3_methods if m in results_dict]

        if inter3_methods:
            print(f"\n  --- P@k for combinations of 3 LLMs (yes/tot) ---")
            # First half
            inter3_methods_1 = inter3_methods[:5]
            print(f"  {'k':<6}", end="")
            for method in inter3_methods_1:
                print(f" | {method:<16}", end="")
            print()
            print("  " + "-"*(7 + 19 * len(inter3_methods_1)))

            for k in [10, 50, 100, 200, 300, 500]:
                if k not in config.k_values:
                    continue
                row = f"  {k:<6}"
                for method in inter3_methods_1:
                    df_res = results_dict[method]
                    df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                    if len(df_t) > 0:
                        p = df_t["P@k"].values[0]
                        count = df_t["count"].values[0]
                        total = df_t["total_positives"].values[0]
                        row += f" | {p:.3f} ({count:>3}/{total:<3}) "
                    else:
                        row += " | -                "
                print(row)

            # Second half
            inter3_methods_2 = inter3_methods[5:]
            if inter3_methods_2:
                print(f"\n  {'k':<6}", end="")
                for method in inter3_methods_2:
                    print(f" | {method:<16}", end="")
                print()
                print("  " + "-"*(7 + 19 * len(inter3_methods_2)))

                for k in [10, 50, 100, 200, 300, 500]:
                    if k not in config.k_values:
                        continue
                    row = f"  {k:<6}"
                    for method in inter3_methods_2:
                        df_res = results_dict[method]
                        df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                        if len(df_t) > 0:
                            p = df_t["P@k"].values[0]
                            count = df_t["count"].values[0]
                            total = df_t["total_positives"].values[0]
                            row += f" | {p:.3f} ({count:>3}/{total:<3}) "
                        else:
                            row += " | -                "
                    print(row)


def print_coverage_table(coverage_dict: Dict[str, pd.DataFrame], config: Config):
    """Coverage table with count/total."""
    print("\n" + "="*180)
    print("POSITIVE COVERAGE PER METHOD (5 LLMs) - yes retrieved / yes total")
    print("="*180)

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4', 'LLM_Inter5']
    main_methods = [m for m in main_methods if m in coverage_dict]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")

        df_first = next(iter(coverage_dict.values()))
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue

        total_pos = df_t["total_positives"].iloc[0]
        n_samples = df_t["n_samples"].iloc[0]
        prevalence = df_t["prevalence"].iloc[0]

        print(f"  Total positives (YES): {total_pos} / {n_samples} (prevalence: {prevalence:.3f})")
        print(f"{'='*180}")

        print(f"\n  {'k':<6}", end="")
        for method in main_methods:
            print(f" | {method[:16]:<16}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in main_methods:
            print(f" | yes/tot (R@k)   ", end="")
        print()
        print("  " + "-"*(7 + 19 * len(main_methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in main_methods:
                df_cov = coverage_dict[method]
                df_tk = df_cov[(df_cov["target"] == target) & (df_cov["k"] == k)]
                if len(df_tk) > 0:
                    retrieved = int(df_tk["positives_retrieved"].values[0])
                    total = int(df_tk["total_positives"].values[0])
                    recall = df_tk["coverage"].values[0]
                    row += f" | {retrieved:>3}/{total:<3} ({recall:.2f}) "
                else:
                    row += " | -                "
            print(row)


def print_detailed_count_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Detailed table with counts only (yes retrieved / yes total)."""
    print("\n" + "="*180)
    print("DETAIL: NUMBER OF YES RETRIEVED / NUMBER OF YES TOTAL (5 LLMs)")
    print("="*180)

    methods = [m for m in results_dict.keys() if m != "Random"]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")
        print(f"{'='*180}")

        df_first = results_dict[methods[0]]
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue
        total_pos = df_t["total_positives"].iloc[0]
        print(f"  Total number of YES: {total_pos}")

        # Header (main methods)
        main_methods = ["TF-IDF", "BM25", "CrossEncoder", "Heuristic",
                       "LLM_Union", "LLM_Inter2", "LLM_Inter3", "LLM_Inter4", "LLM_Inter5"]
        main_methods = [m for m in main_methods if m in results_dict]

        print(f"\n  {'k':<6} | {'Random':<12}", end="")
        for method in main_methods:
            print(f" | {method[:12]:<12}", end="")
        print()
        print("  " + "-"*(7 + 15 + 15 * len(main_methods)))

        for k in config.k_values:
            row = f"  {k:<6}"

            # Random
            df_rand = results_dict["Random"]
            df_tk = df_rand[(df_rand["target"] == target) & (df_rand["k"] == k)]
            if len(df_tk) > 0:
                count = int(df_tk["count"].values[0])
                row += f" | {count:>4}/{total_pos:<4}   "
            else:
                row += " | -            "

            # Other methods
            for method in main_methods:
                df_res = results_dict[method]
                df_tk = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_tk) > 0:
                    count = int(df_tk["count"].values[0])
                    row += f" | {count:>4}/{total_pos:<4}  "
                else:
                    row += " | -            "
            print(row)


def print_agree_disagree_comparison(df_agree_disagree: pd.DataFrame, config: Config):
    """Print the performance comparison on agree vs disagree cases."""
    print("\n" + "="*180)
    print("AGREE vs DISAGREE COMPARISON - Do the methods perform differently on the hard cases?")
    print("="*180)

    if df_agree_disagree.empty:
        print("  No data available.")
        return

    for base_target in ["Gold", "A1", "A2"]:
        df_base = df_agree_disagree[df_agree_disagree["base_target"] == base_target]
        if df_base.empty:
            continue

        print(f"\n{'='*180}")
        print(f"TARGET: {base_target}")
        print(f"{'='*180}")

        df_agree = df_base[df_base["agreement"] == "Agree"]
        df_disagree = df_base[df_base["agreement"] == "Disagree"]

        if not df_agree.empty:
            n_agree = df_agree["n_samples"].iloc[0]
            n_pos_agree = df_agree["total_positives"].iloc[0]
            prev_agree = df_agree["prevalence"].iloc[0]
            print(f"  AGREE:    {n_agree} samples, {n_pos_agree} positives (prevalence: {prev_agree:.3f})")

        if not df_disagree.empty:
            n_disagree = df_disagree["n_samples"].iloc[0]
            n_pos_disagree = df_disagree["total_positives"].iloc[0]
            prev_disagree = df_disagree["prevalence"].iloc[0]
            print(f"  DISAGREE: {n_disagree} samples, {n_pos_disagree} positives (prevalence: {prev_disagree:.3f})")

        print(f"\n  --- Average Precision (AP) ---")
        print(f"  {'METHOD':<22} | {'AP Agree':<10} | {'AP Disagree':<12} | {'Δ (Agree-Disagree)':<18} | {'Δ vs Random Agree':<18} | {'Δ vs Random Disagree':<20}")
        print("  " + "-"*120)

        methods = df_base["method"].unique()

        random_ap_agree = prev_agree if not df_agree.empty else 0
        random_ap_disagree = prev_disagree if not df_disagree.empty else 0

        for method in methods:
            df_m_agree = df_agree[df_agree["method"] == method]
            df_m_disagree = df_disagree[df_disagree["method"] == method]

            ap_agree = df_m_agree["AP"].iloc[0] if not df_m_agree.empty else np.nan
            ap_disagree = df_m_disagree["AP"].iloc[0] if not df_m_disagree.empty else np.nan

            if pd.notna(ap_agree) and pd.notna(ap_disagree):
                delta = ap_agree - ap_disagree
                delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
                delta_rand_agree = ap_agree - random_ap_agree
                delta_rand_disagree = ap_disagree - random_ap_disagree
                delta_rand_agree_str = f"+{delta_rand_agree:.4f}" if delta_rand_agree >= 0 else f"{delta_rand_agree:.4f}"
                delta_rand_disagree_str = f"+{delta_rand_disagree:.4f}" if delta_rand_disagree >= 0 else f"{delta_rand_disagree:.4f}"
            else:
                delta_str = "-"
                delta_rand_agree_str = "-"
                delta_rand_disagree_str = "-"

            ap_agree_str = f"{ap_agree:.4f}" if pd.notna(ap_agree) else "-"
            ap_disagree_str = f"{ap_disagree:.4f}" if pd.notna(ap_disagree) else "-"

            print(f"  {method:<22} | {ap_agree_str:<10} | {ap_disagree_str:<12} | {delta_str:<18} | {delta_rand_agree_str:<18} | {delta_rand_disagree_str:<20}")

        print(f"\n  --- P@k Comparison (Agree vs Disagree) ---")
        key_methods = ["Heuristic", "LLM_Inter5", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2", "CrossEncoder"]
        key_methods = [m for m in key_methods if m in methods]

        for method in key_methods:
            print(f"\n  {method}:")
            print(f"    {'k':<6} | {'P@k Agree (yes/tot)':<22} | {'P@k Disagree (yes/tot)':<24} | {'Δ P@k':<10}")
            print("    " + "-"*75)

            for k in [10, 50, 100, 200, 300, 500]:
                if k not in config.k_values:
                    continue

                df_m_agree_k = df_agree[(df_agree["method"] == method) & (df_agree["k"] == k)]
                df_m_disagree_k = df_disagree[(df_disagree["method"] == method) & (df_disagree["k"] == k)]

                if not df_m_agree_k.empty and not df_m_disagree_k.empty:
                    p_agree = df_m_agree_k["P@k"].values[0]
                    p_disagree = df_m_disagree_k["P@k"].values[0]
                    c_agree = df_m_agree_k["count"].values[0]
                    c_disagree = df_m_disagree_k["count"].values[0]
                    t_agree = df_m_agree_k["total_positives"].values[0]
                    t_disagree = df_m_disagree_k["total_positives"].values[0]
                    delta_p = p_agree - p_disagree
                    delta_str = f"+{delta_p:.3f}" if delta_p >= 0 else f"{delta_p:.3f}"

                    print(f"    {k:<6} | {p_agree:.3f} ({c_agree:>3}/{t_agree:<3})         | {p_disagree:.3f} ({c_disagree:>3}/{t_disagree:<3})           | {delta_str}")
                else:
                    print(f"    {k:<6} | -                      | -                        | -")


def print_false_positive_analysis(df_fp: pd.DataFrame, config: Config):
    """Print the false-positive analysis."""
    print("\n" + "="*180)
    print("FALSE-POSITIVE ANALYSIS - Do the methods limit FP?")
    print("="*180)

    if df_fp.empty:
        print("  No data available.")
        return

    target = "Gold"
    df_t = df_fp[df_fp["target"] == target]
    if df_t.empty:
        print(f"  No data for target={target}")
        return

    n_pos = df_t["n_positives"].iloc[0]
    n_neg = df_t["n_negatives"].iloc[0]
    n_total = n_pos + n_neg

    print(f"\n  TARGET: {target}")
    print(f"  Total: {n_total} samples | {n_pos} positives (YES) | {n_neg} negatives (NO)")
    print(f"  Prevalence: {n_pos/n_total:.1%}" if n_total > 0 else "  Prevalence: N/A")

    print(f"\n  --- True Positives (TP) and False Positives (FP) at each k ---")

    methods = ["Heuristic", "LLM_Inter5", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2", "LLM_Union", "CrossEncoder"]
    methods = [m for m in methods if m in df_t["method"].unique()]

    print(f"\n  {'k':<6} | {'Random':<20}", end="")
    for method in methods:
        print(f" | {method:<20}", end="")
    print()

    print(f"  {'':6} | {'TP/FP (FP%)':<20}", end="")
    for _ in methods:
        print(f" | {'TP/FP (FP%)':<20}", end="")
    print()
    print("  " + "-"*(7 + 23 + 23*len(methods)))

    for k in [10, 50, 100, 200, 300, 500]:
        if k not in config.k_values:
            continue

        row = f"  {k:<6}"

        prevalence = n_pos / n_total
        rand_tp = k * prevalence
        rand_fp = k - rand_tp
        rand_fp_pct = rand_fp / k * 100
        row += f" | {rand_tp:>5.0f}/{rand_fp:<5.0f} ({rand_fp_pct:>4.1f}%)"

        for method in methods:
            df_m = df_t[(df_t["method"] == method) & (df_t["k"] == k)]
            if not df_m.empty:
                tp = df_m["TP"].values[0]
                fp = df_m["FP"].values[0]
                fp_pct = df_m["FP_Rate"].values[0] * 100
                row += f" | {tp:>5}/{fp:<5} ({fp_pct:>4.1f}%)"
            else:
                row += " | -                   "
        print(row)

    print(f"\n  --- False-Positive Origin: Agree vs Disagree ---")

    print(f"\n  {'k':<6}", end="")
    for method in methods:
        print(f" | {method:<28}", end="")
    print()

    print(f"  {'':6}", end="")
    for _ in methods:
        print(f" | {'FP_Agree / FP_Disagree':<28}", end="")
    print()
    print("  " + "-"*(7 + 31*len(methods)))

    for k in [50, 100, 200, 300, 500]:
        if k not in config.k_values:
            continue

        row = f"  {k:<6}"
        for method in methods:
            df_m = df_t[(df_t["method"] == method) & (df_t["k"] == k)]
            if not df_m.empty:
                fp = df_m["FP"].values[0]
                fp_agree = df_m["FP_Agree"].values[0]
                fp_disagree = df_m["FP_Disagree"].values[0]
                if fp > 0:
                    pct_disagree = fp_disagree / fp * 100
                    row += f" | {fp_agree:>4} / {fp_disagree:<4} ({pct_disagree:>4.1f}% Disagree)"
                else:
                    row += f" | 0 / 0 (N/A)              "
            else:
                row += " | -                           "
        print(row)

    print(f"\n  --- False-Positive Reduction vs Random ---")

    print(f"\n  {'k':<6}", end="")
    for method in methods:
        print(f" | {method:<18}", end="")
    print()

    print(f"  {'':6}", end="")
    for _ in methods:
        print(f" | {'FP avoided (%red)':<18}", end="")
    print()
    print("  " + "-"*(7 + 21*len(methods)))

    for k in [50, 100, 200, 300, 500]:
        if k not in config.k_values:
            continue

        row = f"  {k:<6}"
        rand_fp = k * (n_neg / n_total)
        for method in methods:
            df_m = df_t[(df_t["method"] == method) & (df_t["k"] == k)]
            if not df_m.empty:
                fp = df_m["FP"].values[0]
                fp_reduction = rand_fp - fp
                if rand_fp > 0:
                    pct_reduction = fp_reduction / rand_fp * 100
                    row += f" | {fp_reduction:>+5.0f} ({pct_reduction:>+5.1f}%)"
                else:
                    row += f" | {fp_reduction:>+5.0f} (N/A)     "
            else:
                row += " | -                 "
        print(row)


# ============================================================
# PLOTS
# ============================================================
def plot_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualization."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple',
        'LLM_Inter4': 'brown', 'LLM_Inter5': 'darkblue', 'Heuristic': 'magenta'
    }

    main_methods = ['Random', 'TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4', 'LLM_Inter5']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in main_methods:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric}")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_inter4_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualization of the 5 combinations of 4 LLMs."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black',
        'Inter4_LQMQ32': 'blue',
        'Inter4_LQMQ3T': 'green',
        'Inter4_LQQ32Q3T': 'red',
        'Inter4_LMQ32Q3T': 'purple',
        'Inter4_QMQ32Q3T': 'orange',
        'LLM_Inter4': 'brown',
        'LLM_Inter5': 'darkblue'
    }

    methods_to_plot = ['Random', 'Inter4_LQMQ32', 'Inter4_LQMQ3T', 'Inter4_LQQ32Q3T',
                       'Inter4_LMQ32Q3T', 'Inter4_QMQ32Q3T', 'LLM_Inter4', 'LLM_Inter5']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in methods_to_plot:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric} (Inter4 Combinations)")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_inter3_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualization of the 10 combinations of 3 LLMs (selection)."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black',
        'Inter3_LQM': 'blue',
        'Inter3_LQQ32': 'green',
        'Inter3_LQQ3T': 'red',
        'Inter3_QMQ32': 'purple',
        'Inter3_MQ32Q3T': 'orange',
        'LLM_Inter3': 'brown',
        'LLM_Inter4': 'darkblue'
    }

    methods_to_plot = ['Random', 'Inter3_LQM', 'Inter3_LQQ32', 'Inter3_LQQ3T',
                       'Inter3_QMQ32', 'Inter3_MQ32Q3T', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in methods_to_plot:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric} (Inter3 Combinations)")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_coverage_curves(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Coverage curves."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'LLM_Inter4': 'brown',
        'LLM_Inter5': 'darkblue', 'Heuristic': 'magenta'
    }

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4', 'LLM_Inter5']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        df_first = next(iter(coverage_dict.values()))
        df_t_first = df_first[df_first["target"] == target].sort_values("k")

        if len(df_t_first) > 0:
            ax.plot(df_t_first["k"], df_t_first["random_expected"],
                   color='black', linestyle='--', linewidth=1.5,
                   label='Random (expected)', alpha=0.7)

            total_pos = df_t_first["total_positives"].iloc[0]
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        for method in main_methods:
            if method not in coverage_dict:
                continue
            df_cov = coverage_dict[method]
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["positives_retrieved"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k")
        ax.set_ylabel("YES retrieved")
        ax.set_title(f"{target} - Num YES retrieved")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_gain_vs_random(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of the relative gain versus chance."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'LLM_Inter4': 'brown',
        'LLM_Inter5': 'darkblue', 'Heuristic': 'magenta'
    }

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4', 'LLM_Inter5']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

        for method in main_methods:
            if method not in coverage_dict:
                continue
            df_cov = coverage_dict[method]
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["delta_vs_random"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k (number of documents)")
        ax.set_ylabel("Gain vs Random (number of YES)")
        ax.set_title(f"{target} - Gain versus chance")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_count_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of the number of YES retrieved per method."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple',
        'LLM_Inter4': 'brown', 'LLM_Inter5': 'darkblue', 'Heuristic': 'magenta'
    }

    main_methods = ['Random', 'TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4', 'LLM_Inter5']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        total_pos = None

        for method in main_methods:
            if method not in results_dict:
                continue
            df_res = results_dict[method]
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["count"],
                       color=colors.get(method, 'gray'),
                       marker='o' if method != 'Random' else '.',
                       linewidth=2 if method != 'Random' else 1,
                       linestyle='--' if method == 'Random' else '-',
                       label=method, alpha=0.7 if method == 'Random' else 1.0)

                if method != 'Random' and total_pos is None:
                    total_pos = df_t["total_positives"].iloc[0]

        if total_pos is not None:
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        ax.set_xlabel("k")
        ax.set_ylabel("Number of YES retrieved")
        ax.set_title(f"{target} - YES retrieved vs k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_agree_disagree_comparison(df_agree_disagree: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot comparing performance on agree vs disagree."""
    if df_agree_disagree.empty:
        return

    df_gold = df_agree_disagree[df_agree_disagree["base_target"] == "Gold"]
    if df_gold.empty:
        return

    methods_to_plot = ["Heuristic", "LLM_Inter5", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2",
                       "LLM_Union", "CrossEncoder", "BM25", "TF-IDF"]
    methods_to_plot = [m for m in methods_to_plot if m in df_gold["method"].unique()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter5': 'darkblue', 'LLM_Inter4': 'brown',
        'LLM_Inter3': 'purple', 'LLM_Inter2': 'red', 'LLM_Union': 'orange',
        'CrossEncoder': 'green', 'BM25': 'cyan', 'TF-IDF': 'blue'
    }

    ax = axes[0]
    df_agree = df_gold[df_gold["agreement"] == "Agree"]
    for method in methods_to_plot:
        df_m = df_agree[df_agree["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["P@k"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    if not df_agree.empty:
        prev = df_agree["prevalence"].iloc[0]
        ax.axhline(y=prev, color='black', linestyle='--', linewidth=1,
                  label=f'Random ({prev:.3f})', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("P@k")
    ax.set_title("Gold - AGREE cases (annotators agree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.0)

    ax = axes[1]
    df_disagree = df_gold[df_gold["agreement"] == "Disagree"]
    for method in methods_to_plot:
        df_m = df_disagree[df_disagree["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["P@k"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    if not df_disagree.empty:
        prev = df_disagree["prevalence"].iloc[0]
        ax.axhline(y=prev, color='black', linestyle='--', linewidth=1,
                  label=f'Random ({prev:.3f})', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("P@k")
    ax.set_title("Gold - DISAGREE cases (annotators disagree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_agree_vs_disagree_delta(df_agree_disagree: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot showing the performance delta between agree and disagree."""
    if df_agree_disagree.empty:
        return

    df_gold = df_agree_disagree[df_agree_disagree["base_target"] == "Gold"]
    if df_gold.empty:
        return

    methods_to_plot = ["Heuristic", "LLM_Inter5", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2",
                       "LLM_Union", "CrossEncoder", "BM25", "TF-IDF"]
    methods_to_plot = [m for m in methods_to_plot if m in df_gold["method"].unique()]

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter5': 'darkblue', 'LLM_Inter4': 'brown',
        'LLM_Inter3': 'purple', 'LLM_Inter2': 'red', 'LLM_Union': 'orange',
        'CrossEncoder': 'green', 'BM25': 'cyan', 'TF-IDF': 'blue'
    }

    df_agree = df_gold[df_gold["agreement"] == "Agree"]
    df_disagree = df_gold[df_gold["agreement"] == "Disagree"]

    for method in methods_to_plot:
        df_m_agree = df_agree[df_agree["method"] == method].sort_values("k")
        df_m_disagree = df_disagree[df_disagree["method"] == method].sort_values("k")

        if not df_m_agree.empty and not df_m_disagree.empty:
            merged = df_m_agree[["k", "P@k"]].merge(
                df_m_disagree[["k", "P@k"]], on="k", suffixes=("_agree", "_disagree")
            )
            merged["delta"] = merged["P@k_agree"] - merged["P@k_disagree"]

            ax.plot(merged["k"], merged["delta"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel("k")
    ax.set_ylabel("Δ P@k (Agree - Disagree)")
    ax.set_title("Gold - P@k difference between AGREE and DISAGREE cases\n(positive = better on Agree, negative = better on Disagree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_false_positive_analysis(df_fp: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot of the false-positive analysis."""
    if df_fp.empty:
        return

    df_gold = df_fp[df_fp["target"] == "Gold"]
    if df_gold.empty:
        return

    methods_to_plot = ["Heuristic", "LLM_Inter5", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2", "CrossEncoder"]
    methods_to_plot = [m for m in methods_to_plot if m in df_gold["method"].unique()]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter5': 'darkblue', 'LLM_Inter4': 'brown',
        'LLM_Inter3': 'purple', 'LLM_Inter2': 'red', 'LLM_Union': 'orange',
        'CrossEncoder': 'green'
    }

    ax = axes[0]
    for method in methods_to_plot:
        df_m = df_gold[df_gold["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    df_first = df_gold[df_gold["method"] == methods_to_plot[0]].sort_values("k")
    if not df_first.empty:
        ax.plot(df_first["k"], df_first["Random_FP_Expected"],
               color='black', linestyle='--', linewidth=1.5, label='Random (expected)', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("Number of False Positives")
    ax.set_title("Gold - False Positives vs k")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    for method in methods_to_plot:
        df_m = df_gold[df_gold["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP_Reduction_vs_Random"] * 100,
                   color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel("k")
    ax.set_ylabel("FP Reduction vs Random (%)")
    ax.set_title("Gold - False-Positive Reduction")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    for method in methods_to_plot:
        df_m = df_gold[df_gold["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["FP_Disagree_Rate"] * 100,
                   color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    ax.set_xlabel("k")
    ax.set_ylabel("% of FP on Disagree cases")
    ax.set_title("Gold - Proportion of FP in the gray zone")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# MAIN
# ============================================================
def main(filter_union: bool = False, compute_crossencoder: bool = True):
    """Main pipeline."""
    print("="*70)
    print("FULLY UNSUPERVISED RANKING PIPELINE (5 LLMs)")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print("="*70)
    print("""
5 LLMs: Llama, Qwen, Mistral, Qwen32B, Qwen3_32B_Thinking

COMBINATIONS TESTED:
---------------------
- Union: at least 1 LLM says yes
- Inter2: at least 2 LLMs say yes
- Inter3: at least 3 LLMs say yes
- Inter4: at least 4 LLMs say yes
- Inter5: all 5 LLMs say yes

Specific combinations of 4 (C(5,4) = 5):
- Inter4_LQMQ32: Llama + Qwen + Mistral + Qwen32B
- Inter4_LQMQ3T: Llama + Qwen + Mistral + Qwen3Think
- Inter4_LQQ32Q3T: Llama + Qwen + Qwen32B + Qwen3Think
- Inter4_LMQ32Q3T: Llama + Mistral + Qwen32B + Qwen3Think
- Inter4_QMQ32Q3T: Qwen + Mistral + Qwen32B + Qwen3Think

Specific combinations of 3 (C(5,3) = 10):
- Inter3_LQM, Inter3_LQQ32, Inter3_LQQ3T, Inter3_LMQ32, Inter3_LMQ3T
- Inter3_LQ32Q3T, Inter3_QMQ32, Inter3_QMQ3T, Inter3_QQ32Q3T, Inter3_MQ32Q3T

TIE-BREAK HANDLING:
------------------------------------
The binary methods (LLM_Union, LLM_Inter2, LLM_Inter3, LLM_Inter4, LLM_Inter5)
produce 0/1 scores creating ties. To break them
deterministically:

  Score_final = Score_LLM + ε × Score_CrossEncoder  (ε = 1e-6)

Documents with the same LLM vote are ordered by their CrossEncoder score.
The continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic) do not have
significant ties and use direct sorting.
""")

    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_unsupervised_5llm{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # Load & prepare
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # LLM intersection analysis
    llm_stats = analyze_llm_intersection_coverage(df)
    print_llm_intersection_analysis(llm_stats)

    # Compute all scores
    print("\n" + "="*70)
    print("COMPUTING RANKINGS")
    print("="*70)

    all_scores = compute_all_ranking_scores(df)

    results_dict = {}
    coverage_dict = {}

    n_methods = len(all_scores) + 1  # +1 for Random
    print(f"\n  [1/{n_methods}] Random baseline...")
    results_dict["Random"] = compute_random_baseline(df, config)

    for i, (method_name, scores) in enumerate(all_scores.items(), 2):
        print(f"  [{i}/{n_methods}] {method_name}...")
        results_dict[method_name] = evaluate_method(df, scores, method_name, config)
        coverage_dict[method_name] = analyze_positive_coverage(df, scores, method_name, config)

    # Reporting
    print_comparison_table(results_dict, config)
    print_coverage_table(coverage_dict, config)
    print_detailed_count_table(results_dict, config)

    # Agree vs Disagree analysis
    print("\n" + "="*70)
    print("AGREE vs DISAGREE ANALYSIS")
    print("="*70)
    df_agree_disagree = analyze_agree_disagree_performance(df, all_scores, config)
    print_agree_disagree_comparison(df_agree_disagree, config)

    # False-positive analysis
    print("\n" + "="*70)
    print("FALSE-POSITIVE ANALYSIS")
    print("="*70)
    df_fp = analyze_false_positives(df, all_scores, config)
    print_false_positive_analysis(df_fp, config)

    # Plots
    plot_comparison(results_dict, config, f"{config.output_dir}/comparison_main.png")
    plot_inter4_comparison(results_dict, config, f"{config.output_dir}/comparison_inter4.png")
    plot_inter3_comparison(results_dict, config, f"{config.output_dir}/comparison_inter3.png")
    plot_coverage_curves(coverage_dict, config, f"{config.output_dir}/coverage.png")
    plot_gain_vs_random(coverage_dict, config, f"{config.output_dir}/gain_vs_random.png")
    plot_count_comparison(results_dict, config, f"{config.output_dir}/count_comparison.png")

    # Plots Agree vs Disagree
    plot_agree_disagree_comparison(df_agree_disagree, config, f"{config.output_dir}/agree_vs_disagree.png")
    plot_agree_vs_disagree_delta(df_agree_disagree, config, f"{config.output_dir}/agree_disagree_delta.png")

    # False-positive plot
    plot_false_positive_analysis(df_fp, config, f"{config.output_dir}/false_positives.png")

    # Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    for name, scores in all_scores.items():
        df[f"score_{name.lower().replace('-', '_')}"] = scores

    df.to_csv(f"{config.output_dir}/df_with_scores_{timestamp}.csv", index=False)

    df_results = pd.concat(list(results_dict.values()), ignore_index=True)
    df_results.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    df_coverage = pd.concat(list(coverage_dict.values()), ignore_index=True)
    df_coverage.to_csv(f"{config.output_dir}/coverage_{timestamp}.csv", index=False)

    df_agree_disagree.to_csv(f"{config.output_dir}/agree_disagree_results_{timestamp}.csv", index=False)

    df_fp.to_csv(f"{config.output_dir}/false_positives_{timestamp}.csv", index=False)

    with open(f"{config.output_dir}/llm_stats_{timestamp}.json", "w") as f:
        json.dump(llm_stats, f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict, coverage_dict, llm_stats, df_agree_disagree, df_fp


if __name__ == "__main__":
    df, results, coverage, llm_stats, agree_disagree, fp_analysis = main(filter_union=False, compute_crossencoder=True)